# CONSORT checklist extraction: LLM extractor → residual classifier → triaged LLM verifier (v34, rebuilt prototype)

**Task.** Label each sentence of a randomized-trial article with the CONSORT 2010 checklist items it reports. A sentence can carry several items. There are 34 target items: 1a, 1b and 2a are excluded, as in the PubMedBERT baseline. Scoring is micro-F1 over (sentence, item) pairs.

This notebook runs the whole v34 system end to end, **from the input files only** (folder `inputs/`). Nothing is read from earlier experiment folders:

```
 article ─► Step 1  LLM extractor (v26 pipeline, train-only prototype)   ─► pseudo-labels + one-sentence rationale per label
             router → article memory → section extractors
        ─► Step 2  enriched representation x                (sentence + context + pseudo-labels + graph + rationale)
        ─► Step 3  PubMedBERT classifier, LLM-prior residual head  ─► P(item | sentence)
        ─► Step 4  triage                                   ─► pairs where LLM and classifier disagree
        ─► Step 5  paragraph-level LLM verifier              ─► keep/remove, add/reject for flagged pairs
             given the extractor's guidance + corpus evidence from TRAIN gold + the classifier probability
        ─► baselines, error accounting, validation selection, test (once)
```

**Results (validation, 10 articles; recomputed in Parts G and BASE):**

| system | P | R | **F1** | macro F1 |
|---|---|---|---|---|
| LLM only (checklist definitions, one call per article) | 0.596 | 0.578 | 0.587 | 0.538 |
| PubMedBERT text-only (sentence + neighbours + section) | 0.737 | 0.697 | 0.716 | 0.601 |
| **Step 1 extractor** (rebuilt prototype) | 0.690 | 0.821 | **0.750** | 0.635 |
| classifier alone (p ≥ 0.5) | 0.792 | 0.703 | 0.745 | 0.609 |
| Step 1 + classifier decides the flagged pairs (its selected setting) | 0.722 | 0.805 | 0.761 | 0.632 |
| **Step 1 + verifier** (selected setting) | 0.736 | 0.795 | **0.765** | 0.635 |

The verifier improves F1 over the Step 1 extractor in 9 of 10 validation articles (ΔF1 +0.015, article-bootstrap 95% CI +0.008 to +0.021). Most of the remaining error is LLM false positives that the classifier never flags, and LLM misses, which almost never get flagged (Part BASE).

## Glossary

| term | meaning |
|---|---|
| **gold** | the human annotation (sentence, item); used only as the training target and for scoring |
| **pseudo-labels** | the Step 1 LLM's (sentence, item) predictions |
| **rationale** | the one-sentence reason the Step 1 extractor writes for each (item, sentence) it extracts |
| **TP / FP / FN** | correct prediction / predicted but not gold / gold but not predicted |
| **cross-fitting** | train articles are scored by classifiers trained on the *other* train articles (5 folds), so their probabilities look like those of unseen articles |
| **residual head** | classifier output = per-item log-odds of gold given the LLM label (the *prior*, estimated on train) + a learned, L2-penalised *correction*. Before training, it reproduces the LLM. |
| **top-K** | the K items (K = 3) with the highest classifier probability for a sentence; they cover 92% of gold pairs |
| **triage** | choosing which (sentence, item) pairs the verifier re-checks: the pairs where the LLM and the classifier disagree |
| **`remove?`** | a flagged pair the LLM assigned but the classifier does not support: not in top-K, or p < `keep_below` |
| **`add?` L1** | the classifier proposes an item (top-K, p ≥ `add_at_least`) for a sentence the LLM labelled with other items |
| **`add?` L2** | the classifier proposes an item (top-K, p ≥ `add_unlabelled_at_least`) for a sentence the LLM did not label at all |
| **triage setting** | the three thresholds (`keep_below`, `add_at_least`, `add_unlabelled_at_least`) plus the confidence gate |
| **superset** | the loosest triage setting (all thresholds 0.5). The verifier runs **once** on it; any stricter setting flags a subset of the same pairs, so it can be scored offline from the cached decisions. |
| **confidence gate** | which verifier confidence levels may change a label: `("high",)` or `("high", "medium")`. Lower-confidence decisions leave the LLM label unchanged. |
| **selected** | the triage setting and gate with the best micro-F1 on validation (grid search; ties go to fewer reviews) |
| **"classifier decides"** | the free alternative to the verifier: the same flagged pairs, but the classifier's verdict is applied (`remove?` → remove, `add?` → add) |
| **"classifier decides @ its selected"** | that alternative at **its own** best setting from the same grid search, rather than the verifier's setting. This makes it a fair comparison. |
| **corpus evidence** | per flagged item: train-gold statistics, annotated examples, and **near-misses** (train sentences where the extractor assigned the item but the annotators did not) |
| **leave-one-article-out** | when the verifier checks a train article, that article is excluded from its own evidence |
| **prompt signature** | a hash of the verifier task spec and settings; it is part of the cache key, so changing the prompt re-runs the verifier |
| **LLM only** | baseline: the same LLM reads the whole article with only the official CONSORT 2010 checklist definitions (no router, prototype, guideline, classifier or verifier) |
| **PubMedBERT text-only** | baseline: the supervised classifier with previous / target / next sentence and section names as input, and no LLM input |
| **test lock** | `test_lock.json` records the selection at the first test run; re-testing with a different selection stops with an error |

**Policy-table columns** (Part G):
- `FP corrected %`: share of the LLM's false positives that were removed.
- `TP lost %`: share of the LLM's correct pairs that were removed.
- `misses recovered %`: share of the LLM's missed gold pairs that were added.
- `removal/addition precision %`: of the changes made, how many were right.
- `item-presence acc %` and `presence fixed/broken vs LLM`: the **compliance view**, per article × item ("is this item reported anywhere?"). How many of the LLM's verdicts became right, or became wrong.
- `reviews`: the number of flagged pairs at that setting.


## Data protocol

- **Split:** `inputs/split_40_10.json` at article level: 30 train / 10 validation / 10 test.
- **Train** is used to build the prototype, fit the classifier and supply corpus evidence. **Validation** is used to select the triage setting and gate. **Test** is scored once, behind a lock.
- **Prototype:** `inputs/consort_prototype_rebuild_v3_1014.json`, rebuilt from **train articles only** (`CONSORT_Prototype_Rebuild_1014.ipynb`). The earlier prototype held gold-labelled sentences from validation and test articles; Part I checks that this one has none.

## How to run

1. Put the six input files in `inputs/` (or set `CONSORT_INPUT_DIR`):

   | file | used by |
   |---|---|
   | `paragraphs_with_sections.csv`, `sentences.csv` | everything (article text, gold labels) |
   | `split_40_10.json` | train / validation / test split |
   | `annotation_guideline_v3_structured.json` | Step 1 extractors and the verifier |
   | `consort_prototype_rebuild_v3_1014.json` | Step 1 extractors and the verifier (train-only prototype) |
   | `CONSORT_2010_Checklist.csv` | the LLM-only baseline |

2. Set `OPENAI_API_KEY` (or type it when prompted). The LLM is `gpt-5.6-terra` through DSPy (`dspy==3.1.3`). Training the classifiers needs a GPU (`torch`, `transformers`); PubMedBERT is downloaded from Hugging Face, or read from a local folder given in `PUBMEDBERT_PATH`.
3. Run all cells. Every stage writes its results to `outputs/` (or `CONSORT_OUTPUT_DIR`) and **skips work that is already there**, so an interrupted run resumes where it stopped. Delete a stage's folder to recompute it.
4. After validation selection (Part G), set `RUN_TEST = True` and run Part H once.

**Cost, roughly:** Step 1 makes about 8 LLM calls per article (50 articles). The verifier makes one call per paragraph with flagged pairs (about 25 per validation article). The LLM-only baseline makes 1 call per article. The classifier trains 3 seeds on all 30 train articles plus 5 cross-fitting folds, and the text-only baseline trains 3 more seeds: 11 fine-tuning runs × 5 epochs, about 10 minutes per run on one GPU.

## 0. Setup

In [1]:
import ast, copy, hashlib, itertools, json, os, random, re, shutil, textwrap, warnings, getpass
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, replace, asdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from tqdm.auto import tqdm

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 120)

# ---- locations (relative to this notebook unless overridden) ----
INPUT_DIR = Path(os.environ.get("CONSORT_INPUT_DIR", "inputs"))       # the six input files (see "How to run")
OUTPUT_DIR = Path(os.environ.get("CONSORT_OUTPUT_DIR", "outputs"))    # everything this notebook writes
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROTOTYPE_FILE = "consort_prototype_rebuild_v3_1014.json"
PUBMEDBERT = os.environ.get("PUBMEDBERT_PATH", "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
LLM_MODEL = "openai/gpt-5.6-terra"

# ---- run flags ----
RUN_STEP1_LLM = True             # run Step 1 for articles not yet in outputs/step1/
RUN_VERIFIER_SPLITS = ["validation"]   # verifier on validation (test is run in Part H)
RUN_PROMPT_DEV = False           # Part F: verifier prompt development on train articles
RUN_BASELINES = True             # Part BASE: LLM-only and PubMedBERT text-only baselines
RUN_TEST = False                 # Part H: set True only after the validation selection (Part G) is final
N_WORKERS = 4                    # parallel articles for LLM calls

INPUT_FILES = ["paragraphs_with_sections.csv", "sentences.csv", "split_40_10.json",
               "annotation_guideline_v3_structured.json", PROTOTYPE_FILE, "CONSORT_2010_Checklist.csv"]
missing = [f for f in INPUT_FILES if not (INPUT_DIR / f).exists()]
assert not missing, f"missing input files in {INPUT_DIR}: {missing}"
for f in INPUT_FILES:
    print(f"{f:45s} sha256 {hashlib.sha256((INPUT_DIR / f).read_bytes()).hexdigest()[:12]}")

paragraphs_with_sections.csv                  sha256 a052585fa084
sentences.csv                                 sha256 66dd85aecc9d
split_40_10.json                              sha256 2c69aef04a8a
annotation_guideline_v3_structured.json       sha256 0f63331c53d5
consort_prototype_rebuild_v3_1014.json        sha256 c0491ed6ba3e
CONSORT_2010_Checklist.csv                    sha256 1c9ec49c5afa


**LLM.** It is configured only if a stage needs it. The key is read from `OPENAI_API_KEY`, or typed when prompted; never paste a key into a cell.

In [2]:
import dspy

NEEDS_LLM = RUN_STEP1_LLM or bool(RUN_VERIFIER_SPLITS) or RUN_PROMPT_DEV or RUN_BASELINES or RUN_TEST
if NEEDS_LLM:
    api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")
    lm = dspy.LM(LLM_MODEL, temperature=1, api_key=api_key, max_retries=10, max_tokens=32000,
                 reasoning_effort="medium")
    dspy.configure(lm=lm)
print("LLM configured:", NEEDS_LLM, "|", LLM_MODEL if NEEDS_LLM else "")

LLM configured: True | openai/gpt-5.6-terra


# Part A — Step 1: the LLM extractor (v26 pipeline)

This is the v26 pipeline that creates the pseudo-labels. All of it runs with `gpt-5.6-terra`, the **base** router (no prompt optimisation), and the train-only prototype + structured annotation guideline:

1. **Section router:** labels each paragraph with one or more canonical sections (title/abstract, introduction, methods, results, discussion, other).
2. **Article memory:** a grounded JSON of cross-section facts (outcomes, arms, sample size, …), each with source sentence IDs. It is context only, never evidence.
3. **Section extractors:** one call per section, over the paragraphs routed to it. Each extractor sees a fixed task spec (item definitions, guideline rules, global rules) and the section's prototype memory (item prototypes, pairwise boundary rules). It returns (item, sid) pairs, each with a score and a **one-sentence rationale**. The rationale becomes part of the classifier input (Part B).

The v26 pipeline also had a conservative verifier (screener, adjudicator, targeted recovery) after the extractors. It changed almost nothing (validation F1 0.690 before and after), so it is **switched off** and its code removed. The final pairs are the extractors' output.

The code is the v26 code **verbatim**, so the prompts are unchanged, except for the rationale field in the extractor's output schema. GEPA optimisation, gold loading and evaluation-only code were removed.

In [4]:
# ---- imports, constants, section normalization (v26 Step 1 code, verbatim; prompts depend on it) ----
import os
import re
import json
import ast
import copy
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import dspy
except ImportError:
    dspy = None
    print("DSPy is not installed in this environment. Install/configure DSPy before running LLM section labeling.")


# -----------------------------
# Data paths
# -----------------------------
DATA_PATH = INPUT_DIR / "paragraphs_with_sections.csv"


SENTENCE_PATH = INPUT_DIR / "sentences.csv"  # optional; not used in Step 1


# -----------------------------
# Column names
# -----------------------------
ARTICLE_ID_COL = "article"


PARAGRAPH_ID_COL = "pid"


PARAGRAPH_TEXT_COL = "text"


# New gold label column:
# - list if paragraph contains CONSORT item(s): item-native section(s)
# - string if paragraph has no CONSORT item: location-based section
GOLD_SECTIONS_COL = "sections"


# Optional older location/section column. Used only for inspection/debugging if present.
OLD_LOCATION_SECTION_COL = "section"


# Long articles can exceed the context window. The labeler uses compact paragraph text.
MAX_CHARS_PER_PARAGRAPH = 1200


CANONICAL_SECTIONS = [
    "title_abstract",
    "introduction",
    "methods",
    "results",
    "discussion",
    "other_information",
]


SECTION_DESCRIPTION = """
Allowed canonical sections:
- title_abstract: title, abstract, structured summary, abstract-like trial summary
- introduction: scientific background, rationale, objectives, hypotheses
- methods: trial design, participants, interventions, outcomes, randomization, blinding, sample size, statistical methods, protocol/method procedures
- results: participant flow, recruitment/follow-up dates when reported as results, baseline data, numbers analyzed, outcome results, ancillary analyses, harms/adverse events
- discussion: limitations, generalizability/applicability, interpretation, implications, conclusions
- other_information: trial registration, protocol availability, funding, conflicts of interest, acknowledgements, author contributions, ethics statements, data availability, references, miscellaneous article metadata
""".strip()


SECTION_TO_ITEMS = {
    "title_abstract": ["1a", "1b"],
    "introduction": ["2a", "2b"],
    "methods": ["3a", "3b", "4a", "4b", "5", "6a", "6b", "7a", "7b", "8a", "8b", "9", "10", "11a", "11b", "12a", "12b"],
    "results": ["13a", "13b", "14a", "14b", "15", "16", "17a", "17b", "18", "19"],
    "discussion": ["20", "21", "22"],
    "other_information": ["23", "24", "25"],
}


def normalize_section_name(raw):
    """Map raw section strings to the six canonical section labels."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return "other_information"

    s = str(raw).strip().lower()
    s = s.strip("[](){}'\" ")
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)

    if not s:
        return "other_information"

    # Already-canonical spellings after replacing spaces with underscores.
    canonical_guess = s.replace(" ", "_")
    if canonical_guess in CANONICAL_SECTIONS:
        return canonical_guess

    # Title / abstract
    if any(k in s for k in ["title", "abstract", "summary", "synopsis"]):
        return "title_abstract"

    # Introduction
    if any(k in s for k in ["introduction", "background", "rationale", "objective", "objectives", "hypothesis", "hypotheses"]):
        return "introduction"

    # Methods
    method_terms = [
        "method", "methods", "material", "materials", "design", "participant", "participants",
        "patient", "patients", "intervention", "outcome", "random", "randomisation",
        "randomization", "blinding", "masking", "sample size", "statistical", "statistics",
        "analysis", "protocol", "eligibility", "allocation", "procedure"
    ]
    if any(k in s for k in method_terms):
        return "methods"

    # Results
    result_terms = [
        "result", "results", "finding", "findings", "participant flow", "baseline",
        "numbers analysed", "numbers analyzed", "outcome", "estimation", "harms",
        "adverse", "recruitment", "follow-up", "ancillary", "subgroup", "table 1"
    ]
    if any(k in s for k in result_terms):
        return "results"

    # Discussion
    discussion_terms = [
        "discussion", "limitation", "limitations", "generalizability", "generalisability",
        "applicability", "interpretation", "conclusion", "conclusions", "implication", "implications"
    ]
    if any(k in s for k in discussion_terms):
        return "discussion"

    # Other information
    other_terms = [
        "funding", "registration", "trial registration", "conflict", "interest",
        "acknowledg", "ethic", "author", "contribution", "data availability",
        "reference", "appendix", "supplement", "protocol availability"
    ]
    if any(k in s for k in other_terms):
        return "other_information"

    return "other_information"


def _try_parse_list_string(value):
    """Parse JSON/Python-list-like strings; return None if not parseable as list."""
    if not isinstance(value, str):
        return None
    v = value.strip()
    if not v:
        return None

    # JSON or Python literal list, e.g. '["methods", "results"]' or "['methods']"
    if (v.startswith("[") and v.endswith("]")) or (v.startswith("(") and v.endswith(")")):
        try:
            parsed = ast.literal_eval(v)
            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)
        except Exception:
            try:
                parsed = json.loads(v)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                return None
    return None


def clean_section_list(value):
    """Standardize a gold or predicted section value to a non-empty list of canonical labels.

    Handles:
    - a canonical string, e.g. "methods"
    - a raw section string, e.g. "Methods"
    - a Python/JSON list, e.g. ["methods", "results"]
    - a string representation of a list, e.g. "['methods', 'results']"
    - comma/semicolon/pipe-separated strings, if present
    """
    if isinstance(value, float) and pd.isna(value):
        value = "other_information"

    parsed = _try_parse_list_string(value)
    if parsed is not None:
        value = parsed

    if isinstance(value, str):
        v = value.strip()
        # Only split if this looks like multiple labels.
        if any(sep in v for sep in [";", "|", ","]):
            parts = re.split(r"\s*[;,|]\s*", v)
            value = [p for p in parts if p]
        else:
            value = [v]

    if not isinstance(value, (list, tuple, set)):
        value = [value]

    cleaned = []
    for x in value:
        sec = normalize_section_name(x)
        if sec in CANONICAL_SECTIONS and sec not in cleaned:
            cleaned.append(sec)

    return cleaned or ["other_information"]

In [5]:
# ---- paragraphs, sentences and the article JSON the prompts read (v26 Step 1 code, verbatim; prompts depend on it) ----
def require_columns(df, required, name="dataframe"):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}. Available columns: {list(df.columns)}")


def load_paragraph_dataset(path=DATA_PATH):
    df = pd.read_csv(path)
    require_columns(df, [ARTICLE_ID_COL, PARAGRAPH_ID_COL, PARAGRAPH_TEXT_COL, GOLD_SECTIONS_COL], "paragraph dataset")

    df = df.copy()
    df[PARAGRAPH_TEXT_COL] = df[PARAGRAPH_TEXT_COL].fillna("").astype(str)

    # Normalize new mixed single/multi-label target.
    df["gold_sections"] = df[GOLD_SECTIONS_COL].apply(clean_section_list)
    df["gold_sections_json"] = df["gold_sections"].apply(json.dumps)

    # Keep optional old/location section only for inspection.
    if OLD_LOCATION_SECTION_COL in df.columns:
        df["old_location_section_norm"] = df[OLD_LOCATION_SECTION_COL].apply(normalize_section_name)
    else:
        df["old_location_section_norm"] = None

    # Stable within-article paragraph numbering used in JSON output to save tokens.
    df = df.sort_values([ARTICLE_ID_COL, PARAGRAPH_ID_COL]).reset_index(drop=True)
    df["pnum"] = df.groupby(ARTICLE_ID_COL).cumcount()

    return df


paragraph_df = load_paragraph_dataset()


def compact_text(text, max_chars=MAX_CHARS_PER_PARAGRAPH):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."


# -----------------------------
# Sentence hierarchy and flexible gold extraction loading for multi-section item classification
# -----------------------------
def find_first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_sentence_dataset(path=SENTENCE_PATH, paragraph_df=paragraph_df):
    """Load sentence dataframe and standardize to article, pnum, pid, sid, sentence_text."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Sentence file not found: {path} (required: it defines the sentence ids and gold labels)")

    sdf = pd.read_csv(path)
    article_col = find_first_existing_col(sdf, [ARTICLE_ID_COL, "article", "article_id", "pmcid", "paper_id", "doc_id"])
    pid_col = find_first_existing_col(sdf, [PARAGRAPH_ID_COL, "paragraph_id", "para_id", "pid"])
    pnum_col = find_first_existing_col(sdf, ["pnum", "paragraph_index", "paragraph_number", "para_num"])
    sid_col = find_first_existing_col(sdf, ["sid", "sentence_id", "sent_id", "sentence_index", "snum"])
    text_col = find_first_existing_col(sdf, ["sentence", "Sentence", "sentence_text", "sent_text", "text", "Text"])

    missing = [name for name, col in {"article": article_col, "sid": sid_col, "sentence_text": text_col}.items() if col is None]
    if missing:
        raise ValueError(f"Could not infer sentence columns {missing}. Available columns: {list(sdf.columns)}")

    out = sdf.copy().rename(columns={article_col: "article", sid_col: "sid", text_col: "sentence_text"})
    out["article"] = out["article"].astype(str)
    out["sid"] = out["sid"].astype(str)
    out["sentence_text"] = out["sentence_text"].fillna("").astype(str).str.strip()
    out = out[out["sentence_text"].str.len() > 0].copy()

    if pid_col is not None and pid_col != "pid":
        out = out.rename(columns={pid_col: "pid"})
    elif "pid" not in out.columns:
        out["pid"] = None

    if pnum_col is not None and pnum_col != "pnum":
        out = out.rename(columns={pnum_col: "pnum"})

    attach = paragraph_df[[ARTICLE_ID_COL, PARAGRAPH_ID_COL, "pnum"]].copy()
    attach = attach.rename(columns={ARTICLE_ID_COL: "article", PARAGRAPH_ID_COL: "pid"})
    attach["article"] = attach["article"].astype(str)

    if "pnum" not in out.columns or out["pnum"].isna().all():
        if out["pid"].notna().any():
            out = out.merge(attach, on=["article", "pid"], how="left")
        else:
            raise ValueError("Sentence dataframe has neither pnum nor paragraph id to attach paragraph numbers.")
    else:
        out["pnum"] = pd.to_numeric(out["pnum"], errors="coerce")
        if out["pnum"].isna().any() and out["pid"].notna().any():
            tmp = out.merge(attach, on=["article", "pid"], how="left", suffixes=("", "_attached"))
            tmp["pnum"] = tmp["pnum"].fillna(tmp.get("pnum_attached"))
            out = tmp.drop(columns=[c for c in ["pnum_attached"] if c in tmp.columns])

    out = out.dropna(subset=["pnum"]).copy()
    out["pnum"] = out["pnum"].astype(int)
    return out[["article", "pnum", "pid", "sid", "sentence_text"]].drop_duplicates().reset_index(drop=True)


sentence_df = load_sentence_dataset(SENTENCE_PATH, paragraph_df)


# -----------------------------
# Build article examples using one canonical JSON input
# -----------------------------
def build_article_sentence_hierarchy_json(article_df, sentence_df):
    """Create compact article -> paragraph -> sentence hierarchy.

    The same JSON is used by Step 1 and Step 2:
    - Step 1 reads all paragraph objects and predicts paragraph-level sections.
    - Step 2 receives filtered subsets of the same objects and predicts item-sid pairs.

    Model-facing hierarchy intentionally uses pnum + sid only.
    """
    sdf = sentence_df.copy()
    sdf["article"] = sdf["article"].astype(str)
    article_id = str(article_df[ARTICLE_ID_COL].iloc[0])
    article_sents = sdf[sdf["article"] == article_id].copy()

    rows = []
    for _, r in article_df.sort_values("pnum").iterrows():
        pnum = int(r["pnum"])
        sent_rows = article_sents[article_sents["pnum"].astype(int) == pnum].copy()
        sentences = []
        if len(sent_rows):
            for _, s in sent_rows.iterrows():
                sentences.append({
                    "sid": str(s["sid"]),
                    "text": compact_text(s["sentence_text"], MAX_CHARS_PER_PARAGRAPH),
                })
        else:
            sentences.append({
                "sid": f"{article_id}::p{pnum}::s0_fallback",
                "text": compact_text(r[PARAGRAPH_TEXT_COL], MAX_CHARS_PER_PARAGRAPH),
            })

        rows.append({
            "pnum": pnum,
            "paragraph_text": compact_text(r[PARAGRAPH_TEXT_COL], MAX_CHARS_PER_PARAGRAPH),
            "sentences": sentences,
        })
    return json.dumps(rows, ensure_ascii=False)

In [6]:
# ---- router (Step 1.1) and article memory (Step 1.2) (v26 Step 1 code, verbatim; prompts depend on it) ----
if dspy is not None:
    # Fixed task specification for the Step 1 router: canonical labels, input/output
    # format, section definitions, and hard formatting rules. This is supplied as a
    # separate InputField value (`fixed_task_spec`), not baked into `instructions`, so
    # GEPA -- which only ever rewrites a signature's `instructions` -- structurally
    # cannot touch it, mirroring how GenericSectionExtractor protects its
    # `fixed_task_spec` from Step 2 extractor optimization. `instructions` below is
    # reduced to just the ambiguous-case routing heuristics GEPA is actually meant to
    # tune, instead of GEPA having to regenerate this entire spec from scratch every
    # reflection round.
    SECTION_ROUTER_FIXED_TASK_SPEC = f"""
Classify each paragraph of a CONSORT randomized trial article into one or more canonical sections.

You must use exactly these canonical labels:
{", ".join(CANONICAL_SECTIONS)}.

INPUT FORMAT:
The article input may be either:
1) a clean paragraph-numbered text format, or
2) a JSON array of paragraph objects with this structure:
   [
     {{
       "pnum": 0,
       "paragraph_text": "...",
       "sentences": [{{"sid": "...", "text": "..."}}]
     }}
   ]
In the JSON format, use pnum as the paragraph identifier. Sentence IDs are provided for later extraction;
section classification is still paragraph-level.

This is a multi-label classification task:
- Most paragraphs should have one section label.
- Some paragraphs may need multiple labels when they contain content belonging to multiple CONSORT sections.
- Use multiple labels only when the paragraph explicitly contains multiple section functions.

{SECTION_DESCRIPTION}

Hard output rules:
- Return valid JSON only. No markdown and no explanation.
- Include every paragraph number present in the article input exactly once.
- Do not repeat paragraph text in the output.
- Each paragraph must have a non-empty "sections" list.
- Use the article order as global context, but classify by the paragraph's actual content.

Output format:
{{
  "article_id": "<article_id>",
  "paragraph_sections": [
    {{"pnum": 0, "sections": ["title_abstract"]}},
    {{"pnum": 1, "sections": ["introduction"]}},
    {{"pnum": 2, "sections": ["methods", "results"]}}
  ]
}}
""".strip()

    class ArticleSectionLabeling(dspy.Signature):
        """
        Decide section membership for each paragraph using the fixed task specification
        supplied in `fixed_task_spec` (canonical labels, input/output format, and section
        definitions -- do not restate or contradict it here).

        Routing heuristics for ambiguous cases:
        - Do not classify tables, figures, or captions as other_information solely because they are tables/figures.
          Classify table/figure captions by content:
          * trial profile, study flow, participant flow, baseline characteristics, outcomes, Kaplan-Meier curves, adverse events, endpoints, efficacy, or safety -> results
          * intervention details, eligibility, randomization, blinding, assessment procedures -> methods
          * funding, competing interests, ethics approval, author contributions, acknowledgments, provenance, protocol availability -> other_information
        - If a paragraph contains CONSORT item evidence from multiple item-native sections, include all relevant sections.
        """

        article_id = dspy.InputField(desc="Article identifier")
        fixed_task_spec = dspy.InputField(
            desc="Frozen router task spec: canonical labels, input/output JSON format, section definitions, and hard formatting rules"
        )
        article_text = dspy.InputField(desc="Canonical article input. Prefer JSON paragraph/sentence hierarchy when available; contains no gold section labels.")
        output_json = dspy.OutputField(desc="Valid JSON object with article_id and paragraph_sections")


    class ArticleSectionLabeler(dspy.Module):
        def __init__(self):
            super().__init__()
            self.label = dspy.Predict(ArticleSectionLabeling)

        def forward(self, article_id, article_text):
            return self.label(
                article_id=str(article_id),
                fixed_task_spec=SECTION_ROUTER_FIXED_TASK_SPEC,
                article_text=article_text,
            )
else:
    ArticleSectionLabeler = None


# Shared cross-section memory for Step 2. This is context only; source_sids are
# provenance for the memory and are never automatically emitted as CONSORT evidence.
ARTICLE_CONTEXT_KEYS = [
    "trial_structure", "participants_setting", "interventions", "outcomes", "sample_size",
    "randomization", "blinding", "analysis", "timeline_flow", "registration_protocol",
]


if dspy is not None:
    ARTICLE_CONTEXT_FIXED_TASK_SPEC = """
Build a compact CONSORT-oriented cross-reference memory from the full article.
Return JSON with exactly these keys:
{
  "trial_structure": [fact, ...],
  "participants_setting": [fact, ...],
  "interventions": [fact, ...],
  "outcomes": [fact, ...],
  "sample_size": [fact, ...],
  "randomization": [fact, ...],
  "blinding": [fact, ...],
  "analysis": [fact, ...],
  "timeline_flow": [fact, ...],
  "registration_protocol": [fact, ...]
}
Each fact must be:
{"value": "short factual phrase", "source_sids": ["exact sid", ...],
 "certainty": "high|medium|low"}

Rules:
- Use only facts explicitly supported by the article; do not infer unstated details.
- source_sids must be copied from the article input and should be the smallest useful support set.
- Keep facts short and cross-reference oriented (e.g., primary outcome identity/timepoint,
  arm mapping, analysis population, planned N, randomization/blinding structure).
- Empty categories are []. Return JSON only.
""".strip()

    class ArticleContextMemory(dspy.Signature):
        article_id = dspy.InputField(desc="Article identifier")
        fixed_task_spec = dspy.InputField(desc="Frozen cross-section memory schema and grounding rules")
        article_text = dspy.InputField(desc="Canonical article paragraph/sentence hierarchy")
        output_json = dspy.OutputField(desc="Grounded cross-section memory JSON")

    class ArticleAnchorExtractor(dspy.Module):
        # Name retained for compatibility with the existing pipeline/GEPA code.
        def __init__(self):
            super().__init__()
            self.extract = dspy.Predict(ArticleContextMemory)

        def forward(self, article_id, article_text):
            return self.extract(
                article_id=str(article_id),
                fixed_task_spec=ARTICLE_CONTEXT_FIXED_TASK_SPEC,
                article_text=article_text,
            )
else:
    ArticleContextMemory = None
    ArticleAnchorExtractor = None


def extract_json_object(text):
    """Best-effort extraction of a JSON object from an LLM response."""
    if isinstance(text, dict):
        return text
    text = str(text or "").strip()
    if not text:
        raise ValueError("Empty output")

    # Remove markdown fences if present.
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text.strip()).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end+1])

    raise ValueError("Could not find valid JSON object")

In [7]:
# ---- prototype + guideline -> per-section extractor specs (Step 1.3) (v26 Step 1 code, verbatim except the rationale field; prompts depend on it) ----
# -----------------------------
# Multi-section gated extraction configuration
# -----------------------------
PROTOTYPE_PATH = INPUT_DIR / PROTOTYPE_FILE   # train-only rebuilt prototype


ANNOTATION_GUIDELINE_PATH = INPUT_DIR / "annotation_guideline_v3_structured.json"


# Active extractors. Add more sections by setting enabled=True.
SECTION_CONFIGS = {
    "introduction": {
        "enabled": True,
        "official_items": SECTION_TO_ITEMS["introduction"],
        "item_policy": "official_only",
        "min_empirical_section_count": 1,
    },
    "methods": {
        "enabled": True,
        "official_items": SECTION_TO_ITEMS["methods"],
        "item_policy": "official_only",
        "min_empirical_section_count": 1,
    },

    # Placeholders for later expansion.
    "title_abstract": {
        "enabled": False,
        "official_items": SECTION_TO_ITEMS["title_abstract"],
        "item_policy": "official_only",
        "min_empirical_section_count": 1,
    },
    "results": {
        "enabled": True,
        "official_items": SECTION_TO_ITEMS["results"],
        "item_policy": "official_only",
        "min_empirical_section_count": 1,
    },
    "discussion": {
        "enabled": True,
        "official_items": SECTION_TO_ITEMS["discussion"],
        "item_policy": "official_only",
        "min_empirical_section_count": 1,
    },
    "other_information": {
        "enabled": True,
        "official_items": SECTION_TO_ITEMS["other_information"],
        "item_policy": "official_only",
        "min_empirical_section_count": 1,
    },
}


ACTIVE_EXTRACTION_SECTIONS = [
    s for s, cfg in SECTION_CONFIGS.items()
    if cfg.get("enabled", False)
]


ACTIVE_EXTRACTION_ITEMS = sorted(
    {item for s in ACTIVE_EXTRACTION_SECTIONS for item in SECTION_CONFIGS[s]["official_items"]},
    key=lambda x: (int(re.match(r"^(\d+)", str(x)).group(1)) if re.match(r"^(\d+)", str(x)) else 999, str(x))
)


# Start from optimized Step 1 if available, but allow GEPA to keep updating it.
LOAD_OPTIMIZED_SECTION_LABELER_FOR_PIPELINE = False


SECTION_LABELER_SAVE_PATH = Path("./saved_programs_0729_v9/optimized_section_labeler_multilabel.json")


# -----------------------------
# Prototype and guideline loading
# -----------------------------
def load_consort_prototype(path=PROTOTYPE_PATH):
    with open(path, "r", encoding="utf-8") as f:
        proto = json.load(f)
    for key in ["metadata", "item_prototypes", "pairwise_boundary_rules", "global_rules"]:
        if key not in proto:
            raise ValueError(f"Prototype JSON missing key: {key}")
    return proto


def load_annotation_guideline(path=ANNOTATION_GUIDELINE_PATH):
    """Load the structured annotation guideline JSON (general rules + per-item rules)."""
    path = Path(path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "general_annotation_rules": [
            "Annotation guideline file was not found at runtime.",
            "Use the CONSORT checklist definitions, item prototypes, global rules, and pairwise boundary rules as the active annotation guidance.",
            "Do not invent evidence. Extract exact sentence IDs from the provided sentence hierarchy only.",
        ],
        "item_specific_rules": {},
    }


# Cap on how many worked examples GEPA may accumulate per section extractor.
# Kept small and enforced in both the initial snapshot and the parser so the
# lever stays a curated set of illustrative cases rather than an unbounded log.
MAX_WORKED_EXAMPLES = 5


def section_prototype_snapshot(proto, section_name, allowed_items):
    """Return only the mutable prototype memory belonging to one extractor."""
    allowed_items = {str(x) for x in allowed_items}

    item_prototypes = {
        item: copy.deepcopy(proto.get("item_prototypes", {}).get(item, {}))
        for item in sorted(allowed_items, key=str)
        if item in proto.get("item_prototypes", {})
    }

    pairwise_rules = {}
    for rule_id, rule in proto.get("pairwise_boundary_rules", {}).items():
        pair = {str(x) for x in rule.get("items", [])}
        if pair and pair.issubset(allowed_items):
            pairwise_rules[rule_id] = copy.deepcopy(rule)

    # Optional pre-seeded worked examples, keyed by section, in the base prototype file.
    worked_examples = [
        copy.deepcopy(ex)
        for ex in proto.get("worked_examples", {}).get(section_name, [])
    ][:MAX_WORKED_EXAMPLES]

    return {
        "section": section_name,
        "allowed_items": sorted(allowed_items, key=str),
        "item_prototypes": item_prototypes,
        "pairwise_boundary_rules": pairwise_rules,
        "section_strategy": [],
        "worked_examples": worked_examples,
    }


# Load the original prototype once and keep an untouched copy for reproducibility.
consort_prototype = load_consort_prototype(PROTOTYPE_PATH)


original_consort_prototype = copy.deepcopy(consort_prototype)


annotation_guideline = load_annotation_guideline(ANNOTATION_GUIDELINE_PATH)


# -----------------------------
# Fixed extractor specification + section-specific mutable prototype memory
# -----------------------------
def _item_sort_key(x):
    m = re.match(r"^(\d+)([a-z]?)$", str(x))
    if not m:
        return (999, str(x))
    return (int(m.group(1)), m.group(2))


def _section_name_variants(section):
    section = normalize_section_name(section)
    variants = {section, section.replace("_", " ")}
    if section == "title_abstract":
        variants.update(["title", "abstract", "title and abstract"])
    if section == "other_information":
        variants.update(["other", "other information", "administrative", "back matter"])
    return {str(x).lower() for x in variants}


def select_items_for_section_from_prototype(proto, section_name, policy="official_only", min_count=1):
    official = set(SECTION_CONFIGS[section_name]["official_items"])
    if policy == "official_only":
        return sorted(official, key=_item_sort_key)

    if policy != "official_plus_empirical":
        raise ValueError(f"Unknown item policy for {section_name}: {policy}")

    selected = set(official)
    target_variants = _section_name_variants(section_name)
    gold_counts = proto.get("metadata", {}).get("gold_section_counts_by_item", {})

    for item, counts in gold_counts.items():
        for sec, n in counts.items():
            if str(sec).lower() in target_variants and int(n) >= int(min_count):
                selected.add(str(item))

    for item, ip in proto.get("item_prototypes", {}).items():
        priors = {str(x).lower() for x in ip.get("section_prior", [])}
        if priors & target_variants:
            selected.add(str(item))

    return sorted(selected, key=_item_sort_key)


def render_item_definitions(items, proto):
    lines = []
    for item in sorted(items, key=_item_sort_key):
        ip = proto.get("item_prototypes", {}).get(item, {})
        lines.append(f"- {item}: {ip.get('definition', '')}")
    return "\n".join(lines)


def render_global_rules_text(proto, max_rules=18):
    rules = proto.get("global_rules", [])[:max_rules]
    return "\n".join(f"- {r}" for r in rules)


def render_section_guideline_text(guideline_obj, allowed_items):
    """Render only the general rules and item-specific rules relevant to allowed_items."""
    allowed_items = sorted({str(x) for x in allowed_items}, key=_item_sort_key)
    item_rules = guideline_obj.get("item_specific_rules", {})

    lines = ["General annotation rules:"]
    for rule in guideline_obj.get("general_annotation_rules", []):
        lines.append(f"- {rule}")

    lines.append("")
    lines.append("Item-specific annotation rules:")
    for item in allowed_items:
        rule = item_rules.get(item)
        if not rule:
            continue
        parts = []
        if rule.get("title"):
            parts.append(rule["title"])
        if rule.get("special_rule"):
            parts.append(f"Special rule: {rule['special_rule']}")
        for guidance in rule.get("cross_item_guidance", []):
            parts.append(f"Cross-item guidance: {guidance}")
        if parts:
            lines.append(f"- {item}: " + " | ".join(parts))

    return "\n".join(lines)


def render_fixed_section_extraction_spec(section_name, allowed_items, proto, guideline_obj):
    """Render the stable task specification supplied to the extractor."""
    official_defs = render_item_definitions(allowed_items, proto)
    global_rules = render_global_rules_text(proto)
    guideline_excerpt = render_section_guideline_text(guideline_obj, allowed_items)

    return f"""
EXTRACTOR SPECIFICATION

Task:
Extract CONSORT item/sentence-ID pairs from sentence hierarchies routed to the {section_name} extractor.

Extractor section:
{section_name}

Allowed item IDs:
{json.dumps(allowed_items)}

Official definitions:
{official_defs}

Input:
selected_paragraphs_json is a JSON array of paragraph objects. Each paragraph contains pnum,
paragraph_text, and sentence objects with sid and text.

Required output JSON:
{{
  "article_id": "<article id>",
  "section": "{section_name}",
  "extractions": [
    {{
      "item": "<allowed CONSORT item id>",
      "pnum": 3,
      "sid": "<sentence id copied exactly from input>",
      "score": 0.0,
      "rationale": "<one sentence: why this sentence satisfies this item>"
    }}
  ]
}}

Output and extraction constraints:
- Return valid JSON only, with no markdown or commentary.
- Output sid, never full sentence text.
- Copy sid exactly from the input; never invent or normalize it.
- Output only the allowed item IDs listed above.
- Do not infer missing evidence or use hidden gold labels.
- Do not annotate bare section headers.
- A sentence may support multiple items only when it explicitly satisfies each item.
- If no evidence exists, return an empty extractions list.
- score must be numeric and between 0 and 1.
- rationale: ONE sentence (at most 30 words) naming the concrete content of the sentence that satisfies the item.

Annotation guideline:
{guideline_excerpt}

Global extraction rules:
{global_rules}
""".strip()


def render_mutable_section_prototype_memory(section_name, allowed_items, proto):
    """The only extractor instruction text GEPA is allowed to evolve."""
    snapshot = section_prototype_snapshot(proto, section_name, allowed_items)
    return (
        "SECTION-SPECIFIC MUTABLE PROTOTYPE MEMORY\n"
        "This is the only extractor component that may be revised during optimization.\n"
        f"Scope is strictly section={section_name} and items={json.dumps(allowed_items)}.\n"
        "Revise only item_prototypes, pairwise_boundary_rules, section_strategy, or worked_examples.\n"
        f"worked_examples is capped at {MAX_WORKED_EXAMPLES} entries; each entry must have "
        "\"snippet\" (short verbatim-style text span), \"correct_items\" (subset of allowed_items), "
        "and \"rationale\" (why those items and not others). Use it for concrete illustrative cases, "
        "not as a place to restate rules already covered by item_prototypes.\n"
        "Do not introduce item IDs outside allowed_items and do not restate or rewrite the fixed task specification.\n\n"
        + json.dumps(snapshot, ensure_ascii=False, indent=2)
    )


SECTION_ALLOWED_ITEMS = {}


SECTION_FIXED_SPECS = {}


SECTION_MUTABLE_PROTOTYPE_PROMPTS = {}


for _section in ACTIVE_EXTRACTION_SECTIONS:
    cfg = SECTION_CONFIGS[_section]
    items = select_items_for_section_from_prototype(
        original_consort_prototype,
        _section,
        policy=cfg.get("item_policy", "official_only"),
        min_count=cfg.get("min_empirical_section_count", 1),
    )
    SECTION_ALLOWED_ITEMS[_section] = items
    SECTION_FIXED_SPECS[_section] = render_fixed_section_extraction_spec(
        _section, items, original_consort_prototype, annotation_guideline
    )
    SECTION_MUTABLE_PROTOTYPE_PROMPTS[_section] = render_mutable_section_prototype_memory(
        _section, items, original_consort_prototype
    )

In [8]:
# ---- pipeline runtime: router -> memory -> extractors -> verification -> recovery (v26 Step 1 code, verbatim; prompts depend on it) ----
# ================================================================
# Step 2 runtime: router -> extractors -> conservative verification -> recovery
# ================================================================
if dspy is not None:
    def make_section_extraction_signature(section_name, mutable_prototype_text):
        """GEPA hook: mutable prototype stays in instructions; task rules stay fixed."""
        try:
            return dspy.Signature(
                "article_id, fixed_task_spec, article_context_json, retry_feedback, selected_paragraphs_json -> output_json",
                instructions=mutable_prototype_text,
            )
        except Exception:
            class SectionExtraction(dspy.Signature):
                article_id = dspy.InputField()
                fixed_task_spec = dspy.InputField()
                article_context_json = dspy.InputField()
                retry_feedback = dspy.InputField()
                selected_paragraphs_json = dspy.InputField()
                output_json = dspy.OutputField()
            SectionExtraction.__doc__ = mutable_prototype_text
            return SectionExtraction

    class GenericSectionExtractor(dspy.Module):
        def __init__(self, section_name, mutable_prototype_text=None, fixed_task_spec=None):
            super().__init__()
            self.section_name = section_name
            self.fixed_task_spec = fixed_task_spec or SECTION_FIXED_SPECS[section_name]
            self.extract = dspy.Predict(make_section_extraction_signature(
                section_name,
                mutable_prototype_text or SECTION_MUTABLE_PROTOTYPE_PROMPTS[section_name],
            ))

        def forward(self, article_id, selected_paragraphs_json, article_context_json="", retry_feedback=""):
            return self.extract(
                article_id=article_id,
                fixed_task_spec=self.fixed_task_spec,
                article_context_json=article_context_json,
                retry_feedback=retry_feedback,
                selected_paragraphs_json=selected_paragraphs_json,
            )


def get_initial_section_labeler_for_pipeline():
    if dspy is None:
        return None
    if LOAD_OPTIMIZED_SECTION_LABELER_FOR_PIPELINE and SECTION_LABELER_SAVE_PATH.exists():
        labeler = ArticleSectionLabeler(); labeler.load(str(SECTION_LABELER_SAVE_PATH))
        print(f"Loaded optimized Step 1 labeler from {SECTION_LABELER_SAVE_PATH}")
        return labeler
    if globals().get("optimized_section_labeler") is not None:
        print("Using in-memory optimized_section_labeler as Step 1 initializer.")
        return optimized_section_labeler
    print("Using base ArticleSectionLabeler as Step 1 initializer.")
    return ArticleSectionLabeler()


# -----------------------------
# Inference / ablation switches
# -----------------------------
ENABLE_ARTICLE_ANCHORS = True


# Step 1's own verifier is off (it changed almost nothing; see Part A). Final pairs = extractor output.
ENABLE_CROSS_SECTION_VERIFIER = False


ENABLE_VERIFIER_ADJUDICATION = False


ENABLE_VERIFIER_RECOVERY_LOOP = False


ENABLE_TARGETED_ROUTER_RECOVERY = False


MAX_VERIFIER_RECOVERY_ROUNDS = 1  # one recovery pass + one final verification pass


ROUTER_MISSING_FALLBACK_ALL_ACTIVE = True


# Conservative suppression policy. REVIEW and low/medium-confidence REMOVE are kept.
VERIFIER_REMOVE_REQUIRES_HIGH_CONFIDENCE = True


VERIFIER_PROTECTED_ITEMS = set()  # e.g. {"17a"} for a protected-item ablation


VERIFIER_ADJACENT_PARAGRAPH_WINDOW = 1


# Leakage boundary: verifier rules are built ONLY from official item definitions plus
# the structured annotation guideline. Prototype examples/cues/empirical metadata and
# all validation gold/feedback are intentionally excluded.
VERIFIER_RULE_SOURCE_POLICY = "official_definition_and_annotation_guideline_only"


def get_initial_anchor_extractor_for_pipeline():
    return ArticleAnchorExtractor() if dspy is not None and ENABLE_ARTICLE_ANCHORS else None


def get_initial_verifier_for_pipeline():
    return None   # Step 1 verifier removed in this notebook (CrossSectionVerifier)


def get_initial_candidate_adjudicator_for_pipeline():
    return None   # Step 1 verifier removed in this notebook (CandidateAdjudicator)


def get_initial_recovery_router_for_pipeline():
    return None   # Step 1 verifier removed in this notebook (TargetedRoutingRecovery)


def parse_article_paragraphs_json(raw):
    rows = json.loads(raw) if isinstance(raw, str) else raw
    out = {}
    for r in rows:
        pnum = int(r["pnum"])
        out[pnum] = {
            "pnum": pnum,
            "paragraph_text": str(r.get("paragraph_text", r.get("text", ""))),
            "sentences": [
                {"sid": str(s.get("sid", "")), "text": str(s.get("text", ""))}
                for s in r.get("sentences", [])
            ],
        }
    return out


def sid_text_lookup_from_hierarchy(article_paragraphs_json):
    lookup = {}
    for pnum, p in parse_article_paragraphs_json(article_paragraphs_json).items():
        for s in p.get("sentences", []):
            lookup[str(s["sid"])] = {"pnum": pnum, "text": s.get("text", "")}
    return lookup


def validate_router_prediction(article_input_json, section_output_json, active_sections=None):
    """Normalize router output and guarantee every article paragraph remains routable."""
    active_sections = list(active_sections or ACTIVE_EXTRACTION_SECTIONS)
    paragraphs = parse_article_paragraphs_json(article_input_json)
    expected = set(paragraphs)
    diagnostics = {"duplicates": [], "missing": [], "unknown_labels": [], "parse_error": None}
    mapping = {p: set() for p in expected}

    try:
        obj = extract_json_object(section_output_json)
        seen = set()
        for row in obj.get("paragraph_sections", []):
            try:
                pnum = int(row.get("pnum"))
            except Exception:
                continue
            if pnum not in expected:
                continue
            if pnum in seen:
                diagnostics["duplicates"].append(pnum)
            seen.add(pnum)
            raw_sections = row.get("sections", [])
            cleaned = clean_section_list(raw_sections)
            unknown = [
                str(x) for x in (raw_sections if isinstance(raw_sections, list) else [raw_sections])
                if normalize_section_name(x) not in CANONICAL_SECTIONS
            ]
            diagnostics["unknown_labels"].extend(unknown)
            mapping[pnum].update(s for s in cleaned if s in active_sections or s in CANONICAL_SECTIONS)
    except Exception as e:
        diagnostics["parse_error"] = repr(e)

    missing = sorted(p for p in expected if not mapping[p])
    diagnostics["missing"] = missing
    if ROUTER_MISSING_FALLBACK_ALL_ACTIVE:
        for pnum in missing:
            mapping[pnum].update(active_sections)
    return {p: sorted(v) for p, v in mapping.items()}, diagnostics


def pred_sections_map_from_section_output(section_output_json):
    """Compatibility helper for metrics/GEPA diagnostics."""
    obj = extract_json_object(section_output_json)
    mapping = {}
    for row in obj.get("paragraph_sections", []) or []:
        try:
            pnum = int(row.get("pnum"))
        except Exception:
            continue
        mapping.setdefault(pnum, set()).update(clean_section_list(row.get("sections", [])))
    return {p: sorted(v) for p, v in mapping.items()}


def selected_paragraphs_for_target_section(article_paragraphs_json, pred_sections_map, target_section):
    target = normalize_section_name(target_section)
    paragraphs = parse_article_paragraphs_json(article_paragraphs_json)
    selected = [p for pnum, p in sorted(paragraphs.items()) if target in set(pred_sections_map.get(pnum, []))]
    return json.dumps(selected, ensure_ascii=False)


def _paragraph_pnums(raw_json):
    try:
        return {int(r["pnum"]) for r in (json.loads(raw_json) if isinstance(raw_json, str) else raw_json)}
    except Exception:
        return set()


def _paragraph_subset_json(article_input_json, pnums):
    paragraphs = parse_article_paragraphs_json(article_input_json)
    return json.dumps([paragraphs[p] for p in sorted(set(pnums)) if p in paragraphs], ensure_ascii=False)


def sanitize_article_context_json(raw_context, article_input_json):
    """Sanitize article-memory provenance; memory remains context, never evidence."""
    if not raw_context:
        return "", []
    valid_sids = set(sid_text_lookup_from_hierarchy(article_input_json))
    events = []
    try:
        obj = extract_json_object(raw_context)
    except Exception as e:
        return "", [{"kind": "context_parse_error", "detail": repr(e)}]

    for k in [k for k in obj if k not in ARTICLE_CONTEXT_KEYS]:
        events.append({"kind": "context_unknown_category", "category": k})
    obj = {k: obj.get(k, []) for k in ARTICLE_CONTEXT_KEYS}

    for category, facts in list(obj.items()):
        if not isinstance(facts, list):
            obj[category] = []
            events.append({"kind": "context_bad_category", "category": category})
            continue
        cleaned_facts = []
        for fact in facts:
            if not isinstance(fact, dict):
                continue
            fact = dict(fact)
            sids = [str(s) for s in fact.get("source_sids", []) if str(s)]
            invalid = [s for s in sids if s not in valid_sids]
            if invalid:
                events.append({"kind": "context_invalid_sid", "category": category, "sids": invalid})
            fact["source_sids"] = [s for s in sids if s in valid_sids]
            if fact.get("certainty") not in {"high", "medium", "low"}:
                fact["certainty"] = "low"
            cleaned_facts.append(fact)
        obj[category] = cleaned_facts
    return json.dumps(obj, ensure_ascii=False), events


def make_section_extractors():
    return {s: GenericSectionExtractor(s) for s in ACTIVE_EXTRACTION_SECTIONS} if dspy is not None else {}


def _parse_extraction_records(raw, section_name, article_input_json):
    allowed = set(SECTION_ALLOWED_ITEMS.get(section_name, []))
    sid_lookup = sid_text_lookup_from_hierarchy(article_input_json)
    records = []
    try:
        obj = extract_json_object(raw)
    except Exception:
        return records
    for ex in obj.get("extractions", []) or []:
        item, sid = str(ex.get("item", "")).strip(), str(ex.get("sid", "")).strip()
        if item not in allowed or sid not in sid_lookup:
            continue
        records.append({
            "item": item,
            "pnum": sid_lookup[sid]["pnum"],
            "sid": sid,
            "score": ex.get("score"),
        })
    return records


def item_to_official_section_map():
    return {
        str(item): section
        for section in ACTIVE_EXTRACTION_SECTIONS
        for item in SECTION_ALLOWED_ITEMS.get(section, [])
    }


# -----------------------------
# Validation-isolated verifier rulebook
# -----------------------------
def _guideline_rule_for_item(item):
    rule = (annotation_guideline.get("item_specific_rules", {}) or {}).get(str(item), {}) or {}
    return {
        "title": str(rule.get("title", "")),
        "special_rule": str(rule.get("special_rule", "")),
        "cross_item_guidance": [str(x) for x in rule.get("cross_item_guidance", []) or []],
    }


def build_verifier_item_rulebook(items):
    """Static rulebook with no validation-derived examples or statistics.

    Allowed sources:
      - item `definition` field used as the checklist definition in this notebook;
      - structured annotation guideline (general + item-specific rules);
      - official section mapping.

    Deliberately NOT used: prototype cues, pairwise empirical rules, worked examples,
    gold-section counts, validation examples, validation errors, or GEPA feedback.
    """
    item_map = item_to_official_section_map()
    items = sorted({str(x) for x in items if str(x) in item_map}, key=_item_sort_key)
    general_rules = [str(x) for x in annotation_guideline.get("general_annotation_rules", []) or []]
    rules = {}
    for item in items:
        ip = (original_consort_prototype.get("item_prototypes", {}) or {}).get(item, {}) or {}
        section = item_map[item]
        peers = {}
        for peer in SECTION_ALLOWED_ITEMS.get(section, []):
            peer_ip = (original_consort_prototype.get("item_prototypes", {}) or {}).get(str(peer), {}) or {}
            peers[str(peer)] = str(peer_ip.get("definition", ""))
        rules[item] = {
            "item": item,
            "official_section": section,
            "definition": str(ip.get("definition", "")),
            "annotation_rule": _guideline_rule_for_item(item),
            "same_section_item_definitions": peers,
        }
    return {
        "source_policy": VERIFIER_RULE_SOURCE_POLICY,
        "general_annotation_rules": general_rules,
        "items": rules,
    }


def verifier_rule_for_item_json(item):
    book = build_verifier_item_rulebook([item])
    return json.dumps({
        "source_policy": book["source_policy"],
        "general_annotation_rules": book["general_annotation_rules"],
        "item_rule": book["items"].get(str(item), {}),
    }, ensure_ascii=False)


def assert_verifier_rulebook_is_validation_isolated(rulebook):
    """Structural guard against accidentally feeding empirical/validation fields."""
    forbidden_keys = {
        "worked_examples", "gold_section_counts_by_item", "validation_errors",
        "validation_examples", "tp", "fp", "fn", "precision", "recall", "f1",
        "gepa_feedback", "error_logs",
    }
    found = []
    def walk(x, path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                if str(k).lower() in forbidden_keys:
                    found.append(path + "." + str(k))
                walk(v, path + "." + str(k))
        elif isinstance(x, list):
            for i, v in enumerate(x):
                walk(v, path + f"[{i}]")
    walk(rulebook)
    if found:
        raise ValueError(f"Verifier rulebook contains forbidden validation/empirical fields: {found}")
    return True


# -----------------------------
# Candidate context construction
# -----------------------------
def _local_context_for_pnum(article_input_json, pnum, adjacent_window=1):
    paragraphs = parse_article_paragraphs_json(article_input_json)
    ordered = sorted(paragraphs)
    if pnum not in paragraphs:
        return {"current": None, "adjacent": []}
    pos = ordered.index(pnum)
    lo = max(0, pos - int(adjacent_window))
    hi = min(len(ordered), pos + int(adjacent_window) + 1)
    adjacent = [paragraphs[p] for p in ordered[lo:hi] if p != pnum]
    return {"current": paragraphs[pnum], "adjacent": adjacent}


def build_verifier_input_records(section_outputs, article_input_json, suppressed_pairs=None):
    """Unique (item,sid) candidates with producer provenance and full paragraph context."""
    suppressed_pairs = set(suppressed_pairs or [])
    sid_lookup = sid_text_lookup_from_hierarchy(article_input_json)
    pooled = {}
    for section_name, payload in section_outputs.items():
        for ex in _parse_extraction_records(
            payload.get("extraction_output_json", "{}"), section_name, article_input_json
        ):
            key = (ex["item"], ex["sid"])
            if key in suppressed_pairs:
                continue
            rec = pooled.setdefault(key, {
                "item": ex["item"],
                "pnum": ex["pnum"],
                "sid": ex["sid"],
                "text": sid_lookup.get(ex["sid"], {}).get("text", ""),
                "producer_sections": set(),
            })
            rec["producer_sections"].add(section_name)

    records = []
    for idx, key in enumerate(sorted(pooled, key=lambda x: (_item_sort_key(x[0]), str(x[1]))), 1):
        rec = pooled[key]
        rec["producer_sections"] = sorted(rec["producer_sections"])
        rec["candidate_id"] = f"C{idx:04d}"
        rec["paragraph_context"] = _local_context_for_pnum(
            article_input_json, rec["pnum"], VERIFIER_ADJACENT_PARAGRAPH_WINDOW
        )["current"]
        records.append(rec)
    return records


def build_adjudication_candidate(candidate_record, article_input_json):
    """Add adjacent paragraph context only for the small set sent to adjudication."""
    rec = copy.deepcopy(candidate_record)
    rec["local_context"] = _local_context_for_pnum(
        article_input_json, rec["pnum"], VERIFIER_ADJACENT_PARAGRAPH_WINDOW
    )
    return rec


# -----------------------------
# Verifier sanitizers
# -----------------------------
def sanitize_verifier_screen_output(raw_output, candidate_records):
    allowed_items = set(item_to_official_section_map())
    candidate_by_id = {r["candidate_id"]: r for r in candidate_records}
    present_items = {r["item"] for r in candidate_records}
    events, decisions = [], {}

    try:
        obj = extract_json_object(raw_output)
    except Exception as e:
        obj = {}
        events.append({"kind": "verifier_parse_error", "detail": repr(e)})

    for d in obj.get("candidate_decisions", []) or []:
        cid = str(d.get("candidate_id", "")).strip()
        decision = str(d.get("decision", "review")).strip().lower()
        if cid not in candidate_by_id:
            events.append({"kind": "unknown_candidate_id", "candidate_id": cid})
            continue
        if decision not in {"keep", "review", "suspect"}:
            events.append({"kind": "invalid_screen_decision", "candidate_id": cid, "decision": decision})
            decision = "review"
        decisions[cid] = {"decision": decision, "reason": str(d.get("reason", ""))}

    # Omitted decisions default to KEEP: verifier failure cannot destroy recall.
    for cid in candidate_by_id:
        decisions.setdefault(cid, {"decision": "keep", "reason": "default_keep_unmentioned"})

    missing, seen_missing = [], set()
    for m in obj.get("missing_item_candidates", []) or []:
        item = str(m.get("item", "")).strip()
        if item not in allowed_items:
            events.append({"kind": "unknown_item_code", "item": item})
            continue
        if item in present_items:
            events.append({"kind": "missing_item_already_present", "item": item})
            continue
        if item in seen_missing:
            continue
        seen_missing.add(item)
        missing.append({"item": item, "rationale": str(m.get("rationale", ""))})

    return {
        "candidate_decisions": [{"candidate_id": cid, **decisions[cid]} for cid in candidate_by_id],
        "missing_item_candidates": missing,
        "sanitization_events": events,
    }


def sanitize_candidate_adjudication(raw_output, expected_candidate):
    """A candidate is suppressible only after a valid, high-confidence REMOVE."""
    cid = str(expected_candidate["candidate_id"])
    item = str(expected_candidate["item"])
    events = []
    try:
        obj = extract_json_object(raw_output)
    except Exception as e:
        obj = {}
        events.append({"kind": "adjudicator_parse_error", "candidate_id": cid, "detail": repr(e)})

    out_cid = str(obj.get("candidate_id", "")).strip()
    if out_cid != cid:
        events.append({"kind": "adjudicator_candidate_mismatch", "expected": cid, "returned": out_cid})

    decision = str(obj.get("decision", "review")).strip().lower()
    if decision not in {"retain", "remove", "review"}:
        events.append({"kind": "invalid_adjudication_decision", "candidate_id": cid, "decision": decision})
        decision = "review"

    confidence = str(obj.get("confidence", "low")).strip().lower()
    if confidence not in {"high", "medium", "low"}:
        events.append({"kind": "invalid_adjudication_confidence", "candidate_id": cid, "confidence": confidence})
        confidence = "low"

    suppress = (
        decision == "remove"
        and item not in {str(x) for x in VERIFIER_PROTECTED_ITEMS}
        and (confidence == "high" or not VERIFIER_REMOVE_REQUIRES_HIGH_CONFIDENCE)
        and out_cid == cid
    )
    if decision == "remove" and not suppress:
        events.append({
            "kind": "remove_not_applied",
            "candidate_id": cid,
            "item": item,
            "confidence": confidence,
            "protected": item in {str(x) for x in VERIFIER_PROTECTED_ITEMS},
        })

    return {
        "candidate_id": cid,
        "item": item,
        "sid": str(expected_candidate["sid"]),
        "decision": decision,
        "confidence": confidence,
        "reason": str(obj.get("reason", "")),
        "suppress": bool(suppress),
        "sanitization_events": events,
    }


def final_items_from_candidates(candidate_records):
    return [
        {
            "item": r["item"],
            "section": r["producer_sections"][0] if r["producer_sections"] else "",
            "producer_sections": r["producer_sections"],
            "pnum": r["pnum"],
            "sid": r["sid"],
        }
        for r in candidate_records
    ]


def resolve_missing_items_by_section(missing_items):
    item_to_section = item_to_official_section_map()
    out = {s: [] for s in ACTIVE_EXTRACTION_SECTIONS}
    for m in missing_items:
        section = item_to_section.get(str(m.get("item", "")))
        if section in out:
            out[section].append(m)
    return out


def sanitize_recovery_router_output(raw_output, requested_items, omitted_pnums):
    requested_items, omitted_pnums = set(requested_items), set(omitted_pnums)
    picked, events = {item: set() for item in requested_items}, []
    try:
        obj = extract_json_object(raw_output)
    except Exception as e:
        return picked, [{"kind": "recovery_router_parse_error", "detail": repr(e)}]
    for row in obj.get("recovery_candidates", []) or []:
        item = str(row.get("item", "")).strip()
        if item not in requested_items:
            events.append({"kind": "recovery_router_unknown_item", "item": item})
            continue
        for p in row.get("pnums", []) or []:
            try:
                pnum = int(p)
            except Exception:
                continue
            if pnum in omitted_pnums:
                picked[item].add(pnum)
            else:
                events.append({"kind": "recovery_router_invalid_pnum", "item": item, "pnum": pnum})
    return picked, events


def render_targeted_recovery_feedback(missing_items):
    items = sorted({str(m["item"]) for m in missing_items}, key=_item_sort_key)
    return (
        "TARGETED RECOVERY ONLY. Re-check the supplied paragraphs only for these missing items: "
        + ", ".join(items)
        + ". Return normal extraction JSON, but the pipeline will accept NEW evidence only for these items. "
          "Do not revise or remove any previously extracted item because of this recovery call."
    )


def merge_targeted_recovery_patch(section_name, current_output_json, recovery_output_json,
                                    requested_items, article_input_json):
    """Add only requested missing-item evidence; never replace/remove unrelated output."""
    requested = set(requested_items) & set(SECTION_ALLOWED_ITEMS.get(section_name, []))
    current = _parse_extraction_records(current_output_json, section_name, article_input_json)
    proposed = _parse_extraction_records(recovery_output_json, section_name, article_input_json)
    existing = {(r["item"], r["sid"]) for r in current}
    added = []
    for r in proposed:
        key = (r["item"], r["sid"])
        if r["item"] in requested and key not in existing:
            current.append(r); existing.add(key); added.append(r)
    return json.dumps({"extractions": current}, ensure_ascii=False), added


if dspy is not None:
    class MultiSectionGatedPipeline(dspy.Module):
        def __init__(self, section_labeler=None, section_extractors=None, anchor_extractor=None,
                     verifier=None, candidate_adjudicator=None, recovery_router=None):
            super().__init__()
            self.section_labeler = section_labeler or get_initial_section_labeler_for_pipeline()
            self.section_extractors = section_extractors or make_section_extractors()
            self.anchor_extractor = (
                anchor_extractor if anchor_extractor is not None else get_initial_anchor_extractor_for_pipeline()
            )
            self.verifier = verifier if verifier is not None else get_initial_verifier_for_pipeline()
            self.candidate_adjudicator = (
                candidate_adjudicator if candidate_adjudicator is not None
                else get_initial_candidate_adjudicator_for_pipeline()
            )
            self.recovery_router = (
                recovery_router if recovery_router is not None else get_initial_recovery_router_for_pipeline()
            )

        def forward(self, article_id, article_input_json):
            # 1) Route once; validate before any paragraph can disappear.
            section_pred = self.section_labeler(article_id=article_id, article_text=article_input_json)
            section_output_json = getattr(section_pred, "output_json", section_pred)
            pred_map, router_diag = validate_router_prediction(
                article_input_json, section_output_json, self.section_extractors.keys()
            )

            # 2) Article-derived cross-section memory (no gold labels).
            article_context_json, anchor_diag = "", []
            if self.anchor_extractor is not None:
                anchor_pred = self.anchor_extractor(article_id=article_id, article_text=article_input_json)
                raw_context = getattr(anchor_pred, "output_json", anchor_pred)
                article_context_json, anchor_diag = sanitize_article_context_json(raw_context, article_input_json)
            # 3) Independent section extraction.
            section_outputs = {}
            for section_name, extractor in self.section_extractors.items():
                selected_json = selected_paragraphs_for_target_section(
                    article_input_json, pred_map, section_name
                )
                pred = extractor(
                    article_id=article_id,
                    selected_paragraphs_json=selected_json,
                    article_context_json=article_context_json,
                    retry_feedback="",
                )
                section_outputs[section_name] = {
                    "selected_paragraphs_json": selected_json,
                    "extraction_output_json": getattr(pred, "output_json", pred),
                    "recovery_added_pnums": [],
                }

            pre_verifier_records = build_verifier_input_records(section_outputs, article_input_json)
            pre_verifier_pairs = sorted({(r["item"], r["sid"]) for r in pre_verifier_records})

            # Static verifier rules are constructed before any validation/gold scoring occurs.
            initial_rulebook = build_verifier_item_rulebook(r["item"] for r in pre_verifier_records)
            assert_verifier_rulebook_is_validation_isolated(initial_rulebook)

            suppressed_pairs = set()
            verifier_trace = []
            post_pruning_pairs_snapshot = None
            final_verification = {
                "final_items": final_items_from_candidates(pre_verifier_records),
                "screen_decisions": [],
                "adjudications": [],
                "missing_item_candidates": [],
                "removed_items": [],
                "sanitization_events": [],
            }

            if self.verifier is not None:
                for round_idx in range(MAX_VERIFIER_RECOVERY_ROUNDS + 1):
                    candidates = build_verifier_input_records(
                        section_outputs, article_input_json, suppressed_pairs
                    )
                    rulebook = build_verifier_item_rulebook(r["item"] for r in candidates)
                    assert_verifier_rulebook_is_validation_isolated(rulebook)

                    verifier_pred = self.verifier(
                        article_id=article_id,
                        article_context_json=article_context_json,
                        item_rulebook_json=json.dumps(rulebook, ensure_ascii=False),
                        merged_extractions_json=json.dumps(candidates, ensure_ascii=False),
                        allowed_items_json=json.dumps(
                            sorted(item_to_official_section_map(), key=_item_sort_key), ensure_ascii=False
                        ),
                    )
                    raw_screen = getattr(verifier_pred, "output_json", verifier_pred)
                    screen = sanitize_verifier_screen_output(raw_screen, candidates)
                    candidate_by_id = {r["candidate_id"]: r for r in candidates}

                    adjudications = []
                    newly_removed = set()
                    if self.candidate_adjudicator is not None:
                        suspect_ids = [
                            d["candidate_id"] for d in screen["candidate_decisions"]
                            if d["decision"] == "suspect" and d["candidate_id"] in candidate_by_id
                        ]
                        for cid in suspect_ids:
                            candidate = candidate_by_id[cid]
                            adjudication_input = build_adjudication_candidate(candidate, article_input_json)
                            adj_pred = self.candidate_adjudicator(
                                article_id=article_id,
                                article_context_json=article_context_json,
                                item_rule_json=verifier_rule_for_item_json(candidate["item"]),
                                candidate_json=json.dumps(adjudication_input, ensure_ascii=False),
                            )
                            raw_adj = getattr(adj_pred, "output_json", adj_pred)
                            adj = sanitize_candidate_adjudication(raw_adj, candidate)
                            adj["raw_output"] = raw_adj
                            adjudications.append(adj)
                            if adj["suppress"]:
                                newly_removed.add((candidate["item"], candidate["sid"]))

                    suppressed_pairs.update(newly_removed)
                    pruned_candidates = build_verifier_input_records(
                        section_outputs, article_input_json, suppressed_pairs
                    )
                    if post_pruning_pairs_snapshot is None:
                        post_pruning_pairs_snapshot = sorted(
                            {(r["item"], r["sid"]) for r in pruned_candidates}
                        )

                    all_events = list(screen["sanitization_events"])
                    for a in adjudications:
                        all_events.extend(a.get("sanitization_events", []))

                    final_verification = {
                        "final_items": final_items_from_candidates(pruned_candidates),
                        "screen_decisions": screen["candidate_decisions"],
                        "adjudications": adjudications,
                        "missing_item_candidates": screen["missing_item_candidates"],
                        "removed_items": [
                            {"item": i, "sid": s} for i, s in sorted(suppressed_pairs)
                        ],
                        "sanitization_events": all_events,
                    }

                    trace_round = {
                        "round": round_idx,
                        "candidate_count": len(candidates),
                        "candidate_index": [
                            {"candidate_id": r["candidate_id"], "item": r["item"], "sid": r["sid"], "pnum": r["pnum"]}
                            for r in candidates
                        ],
                        "raw_screen_output": raw_screen,
                        "sanitized_screen": screen,
                        "adjudications": adjudications,
                        "removed_pairs": sorted([list(x) for x in newly_removed]),
                        "recovery": [],
                    }

                    # No recovery after the final verification round.
                    if (
                        not ENABLE_VERIFIER_RECOVERY_LOOP
                        or round_idx >= MAX_VERIFIER_RECOVERY_ROUNDS
                        or not screen["missing_item_candidates"]
                    ):
                        verifier_trace.append(trace_round)
                        break

                    missing_by_section = resolve_missing_items_by_section(
                        screen["missing_item_candidates"]
                    )
                    any_added = False
                    for section_name, missing_rows in missing_by_section.items():
                        if not missing_rows:
                            continue
                        requested_items = sorted({str(m["item"]) for m in missing_rows}, key=_item_sort_key)
                        original_pnums = _paragraph_pnums(
                            section_outputs[section_name]["selected_paragraphs_json"]
                        )
                        all_pnums = set(parse_article_paragraphs_json(article_input_json))
                        omitted_pnums = sorted(all_pnums - original_pnums)
                        picked_by_item = {i: set() for i in requested_items}
                        recovery_router_events = []

                        if self.recovery_router is not None and omitted_pnums:
                            omitted_json = _paragraph_subset_json(article_input_json, omitted_pnums)
                            missing_rulebook = build_verifier_item_rulebook(requested_items)
                            assert_verifier_rulebook_is_validation_isolated(missing_rulebook)
                            rr_pred = self.recovery_router(
                                article_id=article_id,
                                missing_items_json=json.dumps(missing_rows, ensure_ascii=False),
                                missing_item_rulebook_json=json.dumps(missing_rulebook, ensure_ascii=False),
                                omitted_paragraphs_json=omitted_json,
                                article_context_json=article_context_json,
                            )
                            raw_rr = getattr(rr_pred, "output_json", rr_pred)
                            picked_by_item, recovery_router_events = sanitize_recovery_router_output(
                                raw_rr, requested_items, omitted_pnums
                            )

                        extra_pnums = sorted(
                            set().union(*picked_by_item.values()) if picked_by_item else set()
                        )
                        retry_pnums = sorted(original_pnums | set(extra_pnums))
                        retry_json = _paragraph_subset_json(article_input_json, retry_pnums)

                        extractor = self.section_extractors[section_name]
                        recovery_pred = extractor(
                            article_id=article_id,
                            selected_paragraphs_json=retry_json,
                            article_context_json=article_context_json,
                            retry_feedback=render_targeted_recovery_feedback(missing_rows),
                        )
                        recovery_output = getattr(recovery_pred, "output_json", recovery_pred)
                        patched_output, added = merge_targeted_recovery_patch(
                            section_name,
                            section_outputs[section_name]["extraction_output_json"],
                            recovery_output,
                            requested_items,
                            article_input_json,
                        )
                        if added:
                            any_added = True
                            section_outputs[section_name]["extraction_output_json"] = patched_output
                            section_outputs[section_name]["selected_paragraphs_json"] = retry_json
                            section_outputs[section_name]["recovery_added_pnums"] = sorted(
                                set(section_outputs[section_name].get("recovery_added_pnums", []))
                                | set(extra_pnums)
                            )

                        trace_round["recovery"].append({
                            "section": section_name,
                            "requested_items": requested_items,
                            "original_pnums": sorted(original_pnums),
                            "extra_pnums": extra_pnums,
                            "added_pairs": [[r["item"], r["sid"]] for r in added],
                            "router_events": recovery_router_events,
                        })

                    verifier_trace.append(trace_round)
                    if not any_added:
                        break

            final_candidates = build_verifier_input_records(
                section_outputs, article_input_json, suppressed_pairs
            )
            final_verification["final_items"] = final_items_from_candidates(final_candidates)
            final_pairs = sorted({(r["item"], r["sid"]) for r in final_candidates})
            if post_pruning_pairs_snapshot is None:
                post_pruning_pairs_snapshot = pre_verifier_pairs

            return dspy.Prediction(
                section_output_json=section_output_json,
                article_context_json=article_context_json,
                section_outputs_json=json.dumps(section_outputs, ensure_ascii=False),
                verification_output_json=json.dumps(final_verification, ensure_ascii=False),
                verifier_trace_json=json.dumps(verifier_trace, ensure_ascii=False),
                pre_verifier_pairs_json=json.dumps(pre_verifier_pairs, ensure_ascii=False),
                post_verifier_pruning_pairs_json=json.dumps(post_pruning_pairs_snapshot, ensure_ascii=False),
                final_verified_pairs_json=json.dumps(final_pairs, ensure_ascii=False),
                router_diagnostics_json=json.dumps(router_diag, ensure_ascii=False),
                anchor_diagnostics_json=json.dumps(anchor_diag, ensure_ascii=False),
            )
else:
    MultiSectionGatedPipeline = None

### Run Step 1

There is one record per article: every JSON field the pipeline returns (router output, memory, section outputs with rationales, and the final pairs). The **final pairs** are the pseudo-labels. Records are appended to `outputs/step1/<split>.jsonl` as articles finish; articles already there are skipped.

In [9]:
SPLIT = json.loads((INPUT_DIR / "split_40_10.json").read_text())
STEP1_DIR = OUTPUT_DIR / "step1"
STEP1_DIR.mkdir(parents=True, exist_ok=True)
STEP1_FIELDS = ["section_output_json", "article_context_json", "section_outputs_json", "verification_output_json",
                "verifier_trace_json", "pre_verifier_pairs_json", "post_verifier_pruning_pairs_json",
                "final_verified_pairs_json", "router_diagnostics_json", "anchor_diagnostics_json"]


def article_input_json(article):
    return build_article_sentence_hierarchy_json(paragraph_df[paragraph_df[ARTICLE_ID_COL].astype(str) == article], sentence_df)


def run_step1_article(article):
    pipeline = MultiSectionGatedPipeline(section_labeler=ArticleSectionLabeler())     # base router, no optimisation
    pred = pipeline(article_id=article, article_input_json=article_input_json(article))
    return {"article": article, "prototype_file": PROTOTYPE_FILE,
            **{f: getattr(pred, f, "") for f in STEP1_FIELDS}}


def load_jsonl(path):
    out = {}
    if Path(path).exists():
        for line in open(path, encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                out[str(r["article"])] = r
    return out


for split in ["train", "validation", "test"]:
    path = STEP1_DIR / f"{split}.jsonl"
    todo = [a for a in SPLIT[split] if a not in load_jsonl(path)]
    if todo and RUN_STEP1_LLM:
        with ThreadPoolExecutor(N_WORKERS) as pool, open(path, "a", encoding="utf-8") as f:
            for fut in tqdm(as_completed([pool.submit(run_step1_article, a) for a in todo]), total=len(todo), desc=split):
                f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n")
    print(f"{split:10s}: {len(load_jsonl(path))}/{len(SPLIT[split])} articles cached"
          + (f"  ({len(todo)} missing; set RUN_STEP1_LLM=True)" if todo and not RUN_STEP1_LLM else ""))

train     : 30/30 articles cached
validation: 10/10 articles cached
test      : 10/10 articles cached


# Part B — Step 2: the enriched input representation

Rows, gold labels, section headings and previous/next sentences come from the **PubMedBERT baseline loader** (copied verbatim below). The classifier's y and the text part of its input are therefore identical to the baseline.

Each sentence also gets:

| group | contents |
|---|---|
| `self` | its own pseudo-labels (35-dim multi-hot, 2a included as an input) and the extractor's score per item |
| `context` | pseudo-labels of the other sentences in its paragraph, and of the previous and next sentences |
| `graph` | for each own item, its **strong** neighbours in a CONSORT item graph (e.g. `6a HAS_RESULT 17a`): similarity to the article's sentences carrying the neighbour, neighbour present, neighbour expected but missing |
| `rationale` (text) | the one-sentence rationale the Step 1 extractor wrote for each pseudo-label (free: part of the extractor output) |
| `extras` | optional signals (router sections, memory citations, position); not used by the final classifier |

The structured groups feed an MLP. The pseudo-labels, rationale and graph evidence are also appended as text segments to the PubMedBERT input.

**Similarity** uses frozen PubMedBERT sentence embeddings, **mean-centred**: raw mean-pooled vectors have a cosine of about 0.95 for any pair.


In [10]:
# ---- PubMedBERT baseline helpers (verbatim from PubMedBERT_CONSORT_Baseline.ipynb) ----
def safe_div(a, b):
    return a / b if b else 0.0


ALL_LABELS = ['1a','1b','2a','2b','3a','3b','4a','4b','5','6a','6b','7a','7b','8a','8b','9','10','11a','11b','12a','12b','13a','13b','14a','14b','15','16','17a','17b','18','19','20','21','22','23','24','25']


LABELS = [x for x in ALL_LABELS if x not in {'1a','1b','2a'}]


@dataclass
class Config:
    model_name: str = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
    revision: str = 'main'  # Set an immutable HF commit for exact reruns; resolved commit is logged.
    seed: int = 42
    epochs: int = 20
    batch_size: int = 4
    encoder_lr: float = 1e-5
    head_lr: float = 1e-3
    dropout: float = 0.1
    max_length: int = 512
    threshold: float = 0.5
    weight_decay: float = 0.0
    device: str = 'auto'
    use_context: bool = True
    use_sections: bool = True
    selection: str = 'final'


def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding='utf-8')


def parse_labels(value):
    if pd.isna(value) or not str(value).strip():
        return []
    labels = list(dict.fromkeys(x.strip() for x in str(value).split(',') if x.strip()))
    unknown = set(labels) - set(ALL_LABELS)
    if unknown: raise ValueError(f'Unknown labels: {unknown}')
    return labels


def load_data(sentences_path, paragraphs_path):
    """Join only observed section metadata. Preserve all rows, including negatives/headings."""
    s = pd.read_csv(sentences_path); p = pd.read_csv(paragraphs_path)
    required_s = {'article','sid','pid','text','target','charStartAt'}
    required_p = {'article','pid','text','section'}
    if not required_s <= set(s): raise ValueError(f'Missing sentence fields: {required_s-set(s)}')
    if not required_p <= set(p): raise ValueError(f'Missing paragraph fields: {required_p-set(p)}')
    repairs = []
    # Attachment-specific typo: verify against paragraph text before correcting.
    bad = s.article.eq('PMC3028348')
    if bad.any():
        if bad.sum() != 1: raise ValueError('Unexpected PMC3028348 rows; inspect before repair')
        r = s.loc[bad].iloc[0]
        q = p[(p.article == 'PMC3028347') & (p.pid == r.pid)]
        if len(q) != 1 or str(r.text).strip() != str(q.iloc[0].text).strip():
            raise ValueError('Article-ID repair evidence does not match')
        repairs.append({'source_index':int(r['index']) if 'index' in r else int(r.name),
                        'sid':r.sid,'old_article':'PMC3028348','new_article':'PMC3028347',
                        'reason':'Exact text and pid match in paragraph file; absent original ID'})
        s.loc[bad,'article'] = 'PMC3028347'
    if s.duplicated(['article','sid']).any(): raise ValueError('Duplicate sentence identifiers')
    if p.duplicated(['article','pid']).any(): raise ValueError('Duplicate paragraph identifiers')
    if s.text.isna().any(): raise ValueError('Missing sentence text')
    s['source_row'] = np.arange(len(s))
    s['all_gold_labels'] = s.target.map(parse_labels)
    s['gold_labels'] = s.all_gold_labels.map(lambda xs: [x for x in xs if x in LABELS])
    # section is the actual broad heading; sections contains normalized categories, not nested headings.
    d = s.merge(p[['article','pid','section']],on=['article','pid'],how='left',validate='many_to_one')
    if d.section.isna().any(): raise ValueError('Missing joined section metadata')
    d['sid_number'] = d.sid.str.extract(r'^S(\d+)$')[0].astype(int)
    d = d.sort_values(['article','sid_number'],kind='stable').reset_index(drop=True)
    # Adjacent sentences can cross paragraphs/sections, but never article boundaries.
    g = d.groupby('article',sort=False)
    for column in ['text','section','sid']:
        d['previous_'+column] = g[column].shift(1).fillna('')
        d['next_'+column] = g[column].shift(-1).fillna('')
    audit = {'sentences':len(d),'articles':d.article.nunique(),'paragraphs':len(p),
             'article_id_repairs':repairs,'labels':LABELS,'excluded_labels':['1a','1b','2a'],
             'unlabeled_original_sentences':int(s.target.isna().sum()),
             'zero_label_sentences_after_exclusion':int(d.gold_labels.map(len).eq(0).sum()),
             'section_limitation':'Broad section only; nested section hierarchy absent in supplied CSV.',
             'file_sha256':{str(Path(f).name):hashlib.sha256(Path(f).read_bytes()).hexdigest()
                            for f in [sentences_path,paragraphs_path]}}
    return d, audit


def targets(d):
    return np.asarray([[float(x in xs) for x in LABELS] for xs in d.gold_labels],dtype=np.float32)


def input_parts(row, cfg):
    def part(section, text):
        return ((str(section).strip()+' ') if cfg.use_sections and text else '')+str(text).strip()
    current = part(row['section'],row['text'])
    if not cfg.use_context: return [current]
    return [part(row['previous_section'],row['previous_text']),current,
            part(row['next_section'],row['next_text'])]


def encode_row(row, tokenizer, cfg):
    """One encoder input, raw CLS representation. Target-first token budgeting is explicit."""
    parts = input_parts(row,cfg)
    pieces = [tokenizer.encode(x,add_special_tokens=False) for x in parts]
    lengths = list(map(len,pieces)); budget = cfg.max_length - 1 - len(pieces)
    if budget < 1: raise ValueError('max_length too small')
    kept = [x[:] for x in pieces]
    if len(kept) == 1: kept[0] = kept[0][:budget]
    elif sum(lengths) > budget:
        # Keep the entire target if it fits; split remaining budget between neighbors.
        kept[1] = kept[1][:budget]; remain = budget-len(kept[1])
        left = min(len(kept[0]),remain//2); right = min(len(kept[2]),remain-left)
        left = min(len(kept[0]),remain-right)
        kept[0] = kept[0][-left:] if left else []
        kept[2] = kept[2][:right]
    ids = [tokenizer.cls_token_id]
    for xs in kept: ids.extend(xs); ids.append(tokenizer.sep_token_id)
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'token_type_ids':[0]*len(ids)}, {
        'original_tokens':sum(lengths)+1+len(pieces),'encoded_tokens':len(ids),
        'target_truncated':len(kept[1 if len(kept)==3 else 0]) < lengths[1 if len(kept)==3 else 0],
        'context_truncated':len(kept)==3 and (len(kept[0])<lengths[0] or len(kept[2])<lengths[2])}


def score(y, probs, threshold=0.5):
    pred = probs >= threshold
    result = {}
    for average in ['micro','macro']:
        pr,rc,f1,_ = precision_recall_fscore_support(y,pred,average=average,zero_division=0)
        result.update({f'{average}_precision':float(pr),f'{average}_recall':float(rc),f'{average}_f1':float(f1)})
    valid = [i for i in range(y.shape[1]) if len(np.unique(y[:,i]))==2]
    result['macro_roc_auc_defined_labels'] = float(np.mean([roc_auc_score(y[:,i],probs[:,i]) for i in valid])) if valid else None
    result['roc_auc_defined_label_count'] = len(valid)
    result['exact_match_accuracy'] = float(np.all(y==pred,axis=1).mean())
    result['threshold'] = threshold
    return result


def seed_everything(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends,'cudnn'):
        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

In [11]:
# ---- evaluation helpers (used from Part B on): pair metrics, policies, selection, bootstrap, examples ----
def pair_metrics(pred, Y, L, articles):
    """Pair-level P/R/F1 + changes relative to the LLM + article-level 'item reported?' compliance view."""
    tp, fp, fn = (pred & Y).sum(), (pred & ~Y).sum(), (~pred & Y).sum()
    P, R = safe_div(tp, tp + fp), safe_div(tp, tp + fn)
    removed, added = L & ~pred, pred & ~L
    m = {"P": P, "R": R, "F1": safe_div(2 * P * R, P + R),
         "macro F1": score(Y.astype(np.float32), pred.astype(np.float32), 0.5)["macro_f1"],
         "FP corrected %": 100 * safe_div((L & ~Y & ~pred).sum(), (L & ~Y).sum()),
         "TP lost %": 100 * safe_div((L & Y & ~pred).sum(), (L & Y).sum()),
         "misses recovered %": 100 * safe_div((~L & Y & pred).sum(), (~L & Y).sum()),
         "removal precision %": 100 * safe_div((removed & ~Y).sum(), removed.sum()),
         "addition precision %": 100 * safe_div((added & Y).sum(), added.sum()),
         "removed": int(removed.sum()), "added": int(added.sum())}
    ua = np.unique(articles)
    g = np.stack([Y[articles == a].any(0) for a in ua])
    p = np.stack([pred[articles == a].any(0) for a in ua])
    l = np.stack([L[articles == a].any(0) for a in ua])
    m.update({"item-presence acc %": 100 * (g == p).mean(),
              "presence fixed vs LLM": int(((l != g) & (p == g)).sum()),
              "presence broken vs LLM": int(((l == g) & (p != g)).sum())})
    return m


def apply_policy(ws, split, setting, decider, decisions=None):
    """Final labels for `split` under a triage `setting`; decider = 'verifier' (cached decisions) or 'classifier'."""
    s = ws.mask(split)
    rem, a1, a2 = triage_flags(ws, setting["keep_below"], setting["add_at_least"], setting["add_unlabelled_at_least"])
    rem, add = rem[s], (a1 | a2)[s]
    pred = ws.llm_label[s].copy()
    if decider == "classifier":
        pred[rem], pred[add] = False, True
        return pred
    gate = set(setting["confidence_gate"])
    sids, arts = ws.frame.loc[s, "sid"].astype(str).to_numpy(), ws.frame.loc[s, "article"].to_numpy()
    for r, j in zip(*np.where(rem | add)):
        d = decisions.get(arts[r], {}).get("decisions", {}).get(f"{LABELS[j]}|{sids[r]}")
        if d and d["confidence"] in gate:
            if rem[r, j] and d["decision"] == "remove":
                pred[r, j] = False
            elif add[r, j] and d["decision"] == "add":
                pred[r, j] = True
    return pred


def n_reviews(ws, split, setting):
    rem, a1, a2 = triage_flags(ws, setting["keep_below"], setting["add_at_least"], setting["add_unlabelled_at_least"])
    return int((rem | a1 | a2)[ws.mask(split)].sum())


def policy_table(ws, split, policies, decisions=None):
    """policies: [(label, setting, decider)] -> metrics table with LLM-alone and classifier-alone rows."""
    s = ws.mask(split)
    Y, L, arts = ws.Y[s], ws.llm_label[s], ws.article_arr[s]
    rows = [{"policy": "LLM alone", **pair_metrics(L, Y, L, arts)},
            {"policy": "classifier alone (p>=0.5)", "reviews": 0, **pair_metrics(np.nan_to_num(ws.probs[s]) >= 0.5, Y, L, arts)}]
    for label, setting, decider in policies:
        rows.append({"policy": label, "reviews": n_reviews(ws, split, setting),
                     **pair_metrics(apply_policy(ws, split, setting, decider, decisions), Y, L, arts)})
    return pd.DataFrame(rows).set_index("policy").round(3)


def select_setting(ws, decisions, decider, split="validation"):
    """Best micro-F1 over GRID on `split` (ties -> fewer reviews). The confidence gate is irrelevant for the classifier."""
    s = ws.mask(split)
    Y, L, arts = ws.Y[s], ws.llm_label[s], ws.article_arr[s]
    scored = []
    for vals in itertools.product(*GRID.values()):
        g = dict(zip(GRID, vals))
        if decider == "classifier" and g["confidence_gate"] != GRID["confidence_gate"][0]:
            continue
        f1 = pair_metrics(apply_policy(ws, split, g, decider, decisions), Y, L, arts)["F1"]
        scored.append((f1, -n_reviews(ws, split, g), g))
    return max(scored, key=lambda x: (x[0], x[1]))[2]


def article_bootstrap(ws, split, preds, comparisons, n_boot=5000, seed=0):
    """Paired article-level bootstrap of micro-F1 differences."""
    s = ws.mask(split)
    Y, arts = ws.Y[s], ws.article_arr[s]
    ua = np.unique(arts)
    idx = [np.where(arts == a)[0] for a in ua]
    counts = lambda p: np.array([[(p[i] & Y[i]).sum(), (p[i] & ~Y[i]).sum(), (~p[i] & Y[i]).sum()] for i in idx])
    f1 = lambda c: 2 * c[0] / max(1, 2 * c[0] + c[1] + c[2])
    boot = np.random.default_rng(seed).integers(0, len(ua), size=(n_boot, len(ua)))
    rows = []
    for a, b in comparisons:
        ca, cb = counts(preds[a]), counts(preds[b])
        d = np.array([f1(ca[x].sum(0)) - f1(cb[x].sum(0)) for x in boot])
        rows.append({"comparison": f"{a} − {b}", "Δ F1": f1(ca.sum(0)) - f1(cb.sum(0)),
                     "95% CI low": np.percentile(d, 2.5), "95% CI high": np.percentile(d, 97.5), "P(Δ ≤ 0)": (d <= 0).mean(),
                     "articles improved": f"{int(sum(f1(x) > f1(y) for x, y in zip(ca, cb)))}/{len(ua)}"})
    return pd.DataFrame(rows).round(4)


OUTCOMES = {
    ("remove", False, True): "FP corrected", ("remove", False, False): "FP kept",
    ("remove", True, True): "TP wrongly removed", ("remove", True, False): "TP correctly kept",
    ("add", True, True): "miss recovered", ("add", True, False): "miss not recovered",
    ("add", False, True): "wrong addition", ("add", False, False): "correctly rejected",
}


def outcome_table(ws, split, setting, decisions):
    """One row per flagged pair: what the verifier did, whether it was right, and whether the classifier was right."""
    rem, a1, a2 = triage_flags(ws, setting["keep_below"], setting["add_at_least"], setting["add_unlabelled_at_least"])
    gate, rows = set(setting["confidence_gate"]), []
    for r in np.where(ws.mask(split))[0]:
        for j in np.where(rem[r] | a1[r] | a2[r])[0]:
            a, sid, item = ws.frame.at[r, "article"], str(ws.frame.at[r, "sid"]), LABELS[j]
            d = decisions.get(a, {}).get("decisions", {}).get(f"{item}|{sid}", {})
            kind = "remove" if rem[r, j] else "add"
            applied = bool(d) and d.get("confidence") in gate and d.get("decision") in ("remove", "add")
            gold = bool(ws.Y[r, j])
            rows.append({"row": int(r), "article": a, "sid": sid, "item": item,
                         "check": "remove?" if kind == "remove" else ("add? L1" if a1[r, j] else "add? L2"),
                         "p": float(ws.probs[r, j]), "gold": gold, "decision": d.get("decision", "missing"),
                         "confidence": d.get("confidence", "missing"), "applied": applied,
                         "outcome": OUTCOMES[(kind, gold, applied)],
                         "verifier_right": (applied != gold) if kind == "remove" else (applied == gold),
                         "classifier_right": (not gold) if kind == "remove" else gold,
                         "rule_basis": d.get("rule_basis", ""), "reason": d.get("reason", "")})
    return pd.DataFrame(rows)


def show_example(ws, x, width=110):
    """Print one flagged pair: sentence, neighbours, labels, classifier top-k, verifier decision and reason."""
    r = x["row"]
    labs = lambda mask: ", ".join(l for l, on in zip(LABELS, mask) if on) or "none"
    top = np.argsort(-ws.probs[r])[:TOP_K]
    print(f"[{x['outcome']}] {x['article']} {x['sid']} | item {x['item']} | {x['check']} | "
          f"verifier: {x['decision']} ({x['confidence']}) | classifier p={x['p']:.2f}")
    print(textwrap.fill("  SENTENCE: " + str(ws.frame.at[r, "text"]), width, subsequent_indent="    "))
    for label, rr in [("prev", r - 1), ("next", r + 1)]:
        if 0 <= rr < len(ws.frame) and ws.frame.at[rr, "article"] == x["article"]:
            print(textwrap.shorten(f"  {label}: {ws.frame.at[rr, 'text']}", width))
    print(f"  section: {ws.frame.at[r, 'section']} | LLM: {labs(ws.llm_label[r])} | gold: {labs(ws.Y[r])} | "
          f"classifier top-{TOP_K}: " + ", ".join(f"{LABELS[j]} {ws.probs[r, j]:.2f}" for j in top))
    print(textwrap.fill("  rule basis: " + str(x["rule_basis"]), width, subsequent_indent="    "))
    print(textwrap.fill("  reason: " + str(x["reason"]), width, subsequent_indent="    "))
    print()

In [12]:
# ---- workspace: rows, split, gold, pseudo-labels, prompt inputs, embeddings ----
EXCLUDED_ITEMS = {"1a", "1b", "2a"}                     # not targets (baseline convention); 2a stays an input signal
ACTIVE_EXTRACTION_ITEMS = sorted(
    {i for s in ["introduction", "methods", "results", "discussion", "other_information"] for i in SECTION_TO_ITEMS[s]},
    key=_item_sort_key)
FEATURE_ITEMS = [x for x in ALL_LABELS if x not in {"1a", "1b"}]   # 35 input-side items (incl. 2a)
FI = {x: i for i, x in enumerate(FEATURE_ITEMS)}
NI = len(FEATURE_ITEMS)
SPLITS = ["train", "validation", "test"]


@dataclass
class Paths:
    """Input files (read-only) and this notebook's output folder."""
    inputs: Path
    out: Path
    prototype_file: str = PROTOTYPE_FILE

    def __post_init__(self):
        i, o = Path(self.inputs), Path(self.out)
        self.sentences = i / "sentences.csv"
        self.paragraphs = i / "paragraphs_with_sections.csv"
        self.split = i / "split_40_10.json"
        self.guideline = i / "annotation_guideline_v3_structured.json"
        self.prototype = i / self.prototype_file
        self.step1_cache = o / "step1"
        self.classifier_cache = o          # features/ and runs/


@dataclass
class Workspace:
    paths: Paths
    frame: pd.DataFrame            # one row per sentence (baseline loader), + split, pnum
    paragraph_df: pd.DataFrame     # v26 paragraph table (pnum numbering used in prompts)
    sentence_df: pd.DataFrame      # v26 sentence table (used to rebuild the exact article JSON)
    llm_records: dict              # split -> article -> Step 1 record
    article_json: dict             # article -> v26 paragraph/sentence hierarchy JSON (verifier input)
    Y: np.ndarray                  # (n, 34) gold labels (targets / scoring only)
    llm_label: np.ndarray          # (n, 34) Step 1 LLM final labels (pseudo-labels)
    emb: np.ndarray                # (n, 768) centred, L2-normalised frozen PubMedBERT sentence embeddings
    prototype: dict = None
    guideline: dict = None
    blocks: dict = None            # Step 2 feature blocks (set by build_features)
    probs: np.ndarray = None       # (n, 34) classifier probabilities (set by load/train classifier)

    @property
    def split_arr(self):
        return self.frame["split"].to_numpy()

    @property
    def article_arr(self):
        return self.frame["article"].to_numpy()

    def mask(self, split):
        return self.split_arr == split


def parse_pairs(raw):
    """[[item, sid], ...] JSON -> {(item, sid)}."""
    try:
        rows = json.loads(raw) if isinstance(raw, str) else (raw or [])
    except Exception:
        return set()
    return {(str(r[0]), str(r[1])) for r in rows if len(r) == 2}


def load_step1_records(cache_dir):
    out = {}
    for split in SPLITS:
        path = Path(cache_dir) / f"{split}.jsonl"
        recs = {}
        if path.exists():
            for line in open(path, encoding="utf-8"):
                if line.strip():
                    r = json.loads(line)
                    recs[str(r["article"])] = r
        out[split] = recs
    return out


def center_normalize(emb):
    """Mean-pooled BERT vectors are anisotropic (cosine ~0.95 for any pair); centre, then L2-normalise."""
    emb = emb - emb.mean(axis=0, keepdims=True)
    return (emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-8, None)).astype(np.float32)


def sentence_embeddings(frame, cache_path, model_name=None, batch_size=64, max_length=128):
    """Frozen PubMedBERT mean-pooled embeddings (raw vectors cached), returned centred + normalised."""
    keys = (frame["article"].astype(str) + "::" + frame["sid"].astype(str)).to_numpy(dtype=str)
    cache_path = Path(cache_path)
    if cache_path.exists():
        z = np.load(cache_path, allow_pickle=False)
        if len(z["keys"]) == len(keys) and (z["keys"] == keys).all():
            return center_normalize(z["emb"])
    import torch
    from transformers import AutoTokenizer, AutoModel
    model_name = model_name or PUBMEDBERT
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok, enc = AutoTokenizer.from_pretrained(model_name), AutoModel.from_pretrained(model_name).to(device).eval()
    texts, out = frame["text"].astype(str).tolist(), []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="sentence embeddings"):
            b = tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=max_length,
                    return_tensors="pt").to(device)
            h, m = enc(**b).last_hidden_state, b["attention_mask"].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1).clamp(min=1.0)).cpu().numpy())
    emb = np.concatenate(out).astype(np.float32)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache_path, keys=keys, emb=emb)
    return center_normalize(emb)


def load_workspace(paths: Paths) -> Workspace:
    """Rows, split, gold, Step 1 pseudo-labels, prompt inputs and embeddings (no LLM calls, no training)."""
    paragraph_df = load_paragraph_dataset(paths.paragraphs)
    sentence_df = load_sentence_dataset(paths.sentences, paragraph_df)
    frame, _ = load_data(paths.sentences, paths.paragraphs)
    split = json.loads(Path(paths.split).read_text())
    role = {a: s for s in SPLITS for a in split[s]}
    frame["split"] = frame["article"].map(role).fillna("unused")
    pnum = {(str(a), int(p)): int(n) for a, p, n in zip(paragraph_df["article"], paragraph_df["pid"], paragraph_df["pnum"])}
    frame["pnum"] = [pnum.get((str(a), int(p)), -1) for a, p in zip(frame["article"], frame["pid"])]

    records = load_step1_records(paths.step1_cache)
    have = {a for recs in records.values() for a in recs}
    frame = frame[frame["article"].isin(have)].reset_index(drop=True)

    emb_all = sentence_embeddings(frame, paths.classifier_cache / "features" / "sentence_embeddings_pubmedbert_meanpool_raw.npz")

    li = {l: i for i, l in enumerate(LABELS)}
    row_of = {(a, s): r for r, (a, s) in enumerate(zip(frame["article"], frame["sid"].astype(str)))}
    llm = np.zeros((len(frame), len(LABELS)), dtype=bool)
    for recs in records.values():
        for a, rec in recs.items():
            for i, s in parse_pairs(rec.get("final_verified_pairs_json")):
                if (a, s) in row_of and i in li:
                    llm[row_of[(a, s)], li[i]] = True

    article_json = {
        a: build_article_sentence_hierarchy_json(paragraph_df[paragraph_df["article"].astype(str) == a], sentence_df)
        for a in sorted(have)
    }
    return Workspace(paths=paths, frame=frame, paragraph_df=paragraph_df, sentence_df=sentence_df,
                     llm_records=records, article_json=article_json, Y=targets(frame).astype(bool),
                     llm_label=llm, emb=emb_all,
                     prototype=load_consort_prototype(paths.prototype),
                     guideline=load_annotation_guideline(paths.guideline))


def router_map(rec):
    """Step 1 router: pnum -> predicted canonical sections."""
    try:
        return pred_sections_map_from_section_output(rec.get("section_output_json") or "{}")
    except Exception:
        return {}


def split_of_article(ws, article):
    return next(s for s, recs in ws.llm_records.items() if article in recs)

In [13]:
ws = load_workspace(Paths(INPUT_DIR, OUTPUT_DIR))
display(ws.frame.groupby("split").agg(articles=("article", "nunique"), sentences=("sid", "size"),
                                      gold_pairs=("gold_labels", lambda s: int(s.map(len).sum()))))
rows = []
for split in ["train", "validation"]:          # test is reported only in Part H
    m = ws.mask(split)
    r = pair_metrics(ws.llm_label[m], ws.Y[m], ws.llm_label[m], ws.article_arr[m])
    rows.append({"split": split, "LLM pairs": int(ws.llm_label[m].sum()), "gold pairs": int(ws.Y[m].sum()),
                 **{k: r[k] for k in ["P", "R", "F1", "macro F1"]}})
print("Step 1 LLM alone (sentence x item pairs, 34 items):")
display(pd.DataFrame(rows).set_index("split").round(3))

,articles,sentences,gold_pairs
split,,,
test,10,2177,1014
train,30,6465,3069
validation,10,2067,972


Step 1 LLM alone (sentence x item pairs, 34 items):


,LLM pairs,gold pairs,P,R,F1,macro F1
split,,,,,,
train,3629,3069,0.684,0.808,0.741,0.775
validation,1156,972,0.690,0.821,0.750,0.635


In [14]:
# ---- Step 2 feature construction ----
# CONSORT item graph (0923_v28). Only "strong" edges are used for cross-references.
CONSORT_GRAPH_V1 = [
    ("2b", "TARGETS", "6a", "strong"), ("2b", "EVALUATED_BY", "17a", "strong"), ("22", "ADDRESSES", "2b", "strong"),
    ("3a", "STRUCTURES", "5", "context"), ("3a", "EXPECTED_ALLOCATION_IN", "13a", "strong"),
    ("3a", "DEFINES_GROUPING_FOR", "15", "context"), ("3a", "DEFINES_GROUPING_FOR", "16", "context"),
    ("3b", "MAY_MODIFY", "4a", "dynamic"), ("3b", "MAY_MODIFY", "5", "dynamic"), ("3b", "MAY_MODIFY", "6a", "dynamic"),
    ("3b", "MAY_MODIFY", "7a", "dynamic"), ("3b", "MAY_MODIFY", "8b", "dynamic"), ("3b", "MAY_MODIFY", "11a", "dynamic"),
    ("3b", "MAY_MODIFY", "12a", "dynamic"),
    ("4a", "CONSTRAINS", "13a", "strong"), ("4a", "DESCRIBED_BY", "15", "strong"), ("4a", "CONTEXT_FOR", "21", "context"),
    ("4b", "CONTEXT_FOR", "14a", "context"), ("4b", "CONTEXT_FOR", "21", "context"),
    ("5", "ASSIGNED_IN", "13a", "strong"), ("5", "EVALUATED_IN", "17a", "strong"), ("5", "EVALUATED_IN", "17b", "strong"),
    ("5", "SOURCE_OF_POTENTIAL", "19", "strong"), ("5", "COMPARES_SIMILARITY_IN", "11b", "strong"),
    ("6a", "ANALYZED_BY", "12a", "strong"), ("6a", "HAS_RESULT", "17a", "strong"), ("6a", "HAS_BINARY_RESULT", "17b", "strong"),
    ("6a", "MAY_HAVE_ANALYSIS", "18", "context"),
    ("6b", "CHANGES", "6a", "strong"), ("6b", "EXPLAINS_DIFFERENCE_IN", "17a", "strong"),
    ("7a", "COMPARE_TARGET_WITH", "13a", "strong"), ("7a", "COMPARE_TARGET_WITH", "16", "context"),
    ("7b", "EXPECTED_STOPPING_RULE_FOR", "14b", "strong"),
    ("8b", "CONSTRAINS", "8a", "strong"), ("8b", "SHOULD_ALIGN_WITH", "3a", "strong"), ("8a", "IMPLEMENTED_BY", "10", "strong"),
    ("9", "USED_DURING", "10", "strong"), ("10", "PRODUCES_ASSIGNMENT_IN", "13a", "context"),
    ("11b", "SUPPORTS", "11a", "context"), ("11b", "COMPARES", "5", "strong"),
    ("12a", "ANALYZES", "6a", "strong"), ("12a", "USES_DENOMINATOR_FROM", "16", "strong"), ("12a", "PRODUCES", "17a", "strong"),
    ("12a", "PRODUCES", "17b", "strong"), ("12b", "METHOD_FOR", "18", "strong"),
    ("13a", "PRECEDES", "13b", "strong"), ("13a", "RECONCILES_WITH", "16", "strong"),
    ("13b", "EXPLAINS_DIFFERENCE_IN", "16", "strong"), ("13b", "MAY_INFORM", "20", "context"),
    ("14a", "TIME_CONTEXT_FOR", "13a", "context"), ("14b", "EXPLAINED_BY", "7b", "strong"),
    ("15", "CHARACTERIZES", "4a", "strong"), ("15", "CONTEXT_FOR", "21", "context"),
    ("16", "DENOMINATOR_FOR", "17a", "strong"), ("16", "DENOMINATOR_FOR", "17b", "strong"),
    ("16", "DENOMINATOR_FOR", "18", "context"), ("16", "DENOMINATOR_FOR", "19", "context"),
    ("17a", "SUPPORTS", "22", "strong"), ("17b", "SPECIALIZES", "17a", "strong"),
    ("18", "QUALIFIES", "22", "strong"), ("19", "QUALIFIES", "22", "strong"),
    ("20", "QUALIFIES", "22", "strong"), ("21", "QUALIFIES", "22", "strong"),
    ("23", "CAN_CROSSCHECK", "2b", "external"), ("23", "CAN_CROSSCHECK", "3a", "external"), ("23", "CAN_CROSSCHECK", "6a", "external"),
    ("24", "CAN_CROSSCHECK", "3b", "external"), ("24", "CAN_CROSSCHECK", "6a", "external"), ("24", "CAN_CROSSCHECK", "6b", "external"),
    ("24", "CAN_CROSSCHECK", "7a", "external"), ("24", "CAN_CROSSCHECK", "7b", "external"), ("24", "CAN_CROSSCHECK", "12a", "external"),
    ("24", "CAN_CROSSCHECK", "12b", "external"),
    ("25", "MAY_DESCRIBE_SUPPORT_FOR", "5", "evidence_only"), ("25", "MAY_DESCRIBE_ROLE_IN", "3a", "evidence_only"),
    ("25", "MAY_DESCRIBE_ROLE_IN", "8a", "evidence_only"), ("25", "MAY_DESCRIBE_ROLE_IN", "10", "evidence_only"),
    ("25", "MAY_DESCRIBE_ROLE_IN", "12a", "evidence_only"),
]
GRAPH_EDGE_STRENGTHS = {"strong"}
GRAPH_TOP_K_TEXT, GRAPH_TEXT_CHARS = 3, 300


def graph_neighbors(strengths=GRAPH_EDGE_STRENGTHS):
    """item -> [{neighbor, edge}] over the chosen edge strengths (undirected lookup, edge text keeps direction)."""
    nbrs = defaultdict(list)
    for s, rel, t, strength in CONSORT_GRAPH_V1:
        if strength in strengths and s in FI and t in FI:
            nbrs[s].append({"neighbor": t, "edge": f"{s} {rel} {t}"})
            nbrs[t].append({"neighbor": s, "edge": f"{s} {rel} {t}"})
    return dict(nbrs)


GRAPH_NEIGHBORS = graph_neighbors()


def multi_hot(items):
    v = np.zeros(NI, dtype=np.float32)
    for x in items:
        if x in FI:
            v[FI[x]] = 1.0
    return v


def fmt_items(items):
    items = sorted({x for x in items if x in FI}, key=_item_sort_key)
    return ", ".join(items) if items else "none"


def llm_scores(rec):
    """Max extractor score per (item, sid) across section extractors; missing/invalid score -> 1.0."""
    scores = {}
    try:
        outputs = json.loads(rec.get("section_outputs_json") or "{}")
    except Exception:
        outputs = {}
    for payload in outputs.values():
        try:
            obj = extract_json_object(payload.get("extraction_output_json", "{}"))
        except Exception:
            continue
        for ex in obj.get("extractions", []) or []:
            key = (str(ex.get("item", "")).strip(), str(ex.get("sid", "")).strip())
            try:
                s = float(ex.get("score"))
            except (TypeError, ValueError):
                s = float("nan")
            s = 1.0 if not np.isfinite(s) else min(max(s, 0.0), 1.0)
            scores[key] = max(scores.get(key, 0.0), s)
    return scores


def extractor_rationales(rec, max_chars=400):
    """(item, sid) -> the extractor's one-sentence rationale (first non-empty one across section extractors)."""
    out = {}
    try:
        outputs = json.loads(rec.get("section_outputs_json") or "{}")
    except Exception:
        outputs = {}
    for payload in outputs.values():
        try:
            obj = extract_json_object(payload.get("extraction_output_json", "{}"))
        except Exception:
            continue
        for ex in obj.get("extractions", []) or []:
            key = (str(ex.get("item", "")).strip(), str(ex.get("sid", "")).strip())
            reason = str(ex.get("rationale", "") or "").strip()
            if reason and key not in out:
                out[key] = reason[:max_chars]
    return out


def memory_categories(rec):
    """sid -> grounded-memory categories that cite it as a source."""
    out = defaultdict(set)
    try:
        obj = json.loads(rec.get("article_context_json") or "{}")
    except Exception:
        obj = {}
    for category in ARTICLE_CONTEXT_KEYS:
        for fact in obj.get(category, []) or []:
            if isinstance(fact, dict):
                for s in fact.get("source_sids", []) or []:
                    out[str(s)].add(category)
    return out


def article_features(sub, emb, rec):
    """Feature blocks + text segments for one article's sentences (rows of `sub`, in order)."""
    n = len(sub)
    sids, pids, pnums = sub["sid"].astype(str).tolist(), sub["pid"].tolist(), sub["pnum"].tolist()
    pos = {s: k for k, s in enumerate(sids)}
    final = parse_pairs(rec.get("final_verified_pairs_json"))
    scores = llm_scores(rec)
    reasons = extractor_rationales(rec)
    memory, router = memory_categories(rec), router_map(rec)

    def by_sid(pairs):
        d = defaultdict(set)
        for i, s in pairs:
            d[s].add(i)
        return d

    own_items = by_sid(final)
    item_rows = defaultdict(list)
    for i, s in sorted(final):       # sorted: the original iterated a set, so tie-breaks varied between runs
        if s in pos:
            item_rows[i].append(pos[s])
    para_rows = defaultdict(list)
    for k, p in enumerate(pids):
        para_rows[p].append(k)
    article_items = multi_hot({i for i, _ in final})

    blocks = {g: [] for g in ["self", "context", "graph", "extras", "rationale"]}
    pseudo_text, graph_text, rationale_text = [], [], []
    for k, sid in enumerate(sids):
        own = own_items.get(sid, set())
        # self: own pseudo-labels + extractor scores
        score_vec = np.zeros(NI, dtype=np.float32)
        for i in sorted(own, key=_item_sort_key):
            if i in FI:
                score_vec[FI[i]] = scores.get((i, sid), 1.0)
        blocks["self"].append(np.concatenate([multi_hot(own), score_vec]))
        # context: other sentences in the paragraph, previous and next sentence
        para_items = set().union(*[own_items.get(sids[m], set()) for m in para_rows[pids[k]] if m != k])
        prev_items = own_items.get(sids[k - 1], set()) if k > 0 else set()
        next_items = own_items.get(sids[k + 1], set()) if k + 1 < n else set()
        blocks["context"].append(np.concatenate([multi_hot(para_items), multi_hot(prev_items), multi_hot(next_items)]))
        # graph: own item i -> strong neighbour j -> this article's j-labelled sentences
        sim_vec, present, expected = (np.zeros(NI, dtype=np.float32) for _ in range(3))
        evidence = {}
        for i in sorted(own, key=_item_sort_key):
            for nb in GRAPH_NEIGHBORS.get(i, []):
                j = nb["neighbor"]
                expected[FI[j]] = 1.0
                cand = [m for m in item_rows.get(j, []) if m != k]
                if not cand:
                    continue
                sims = emb[cand] @ emb[k]
                present[FI[j]] = 1.0
                sim_vec[FI[j]] = max(sim_vec[FI[j]], float(sims.max()))
                for m, sv in zip(cand, sims):
                    if m not in evidence or sv > evidence[m][0]:
                        evidence[m] = (float(sv), j, nb["edge"])
        blocks["graph"].append(np.concatenate([sim_vec, present, expected * (1.0 - present)]))
        top = sorted(evidence.items(), key=lambda kv: -kv[1][0])[:GRAPH_TOP_K_TEXT]
        graph_text.append(" ; ".join(
            f"[{edge} | sim {sv:.2f}] {compact_text(sub['text'].iloc[m], GRAPH_TEXT_CHARS)}"
            for m, (sv, j, edge) in top) or "graph evidence: none")
        # extras (optional group, not used by the final classifier): article items, router, memory, position
        router_sections = set(router.get(int(pnums[k]), [])) if pnums[k] >= 0 else set()
        blocks["extras"].append(np.concatenate([
            article_items,
            np.array([float(s in router_sections) for s in CANONICAL_SECTIONS], dtype=np.float32),
            np.array([float(c in memory.get(sid, set())) for c in ARTICLE_CONTEXT_KEYS], dtype=np.float32),
            np.array([k / max(1, n - 1), (pnums[k] / max(1, max(pnums))) if pnums[k] >= 0 else 0.0], dtype=np.float32),
        ]))
        # rationale: structured fields exist only for 'dedicated' rationales (not used here) -> zeros
        blocks["rationale"].append(np.zeros(4 * NI, dtype=np.float32))
        pseudo_text.append(f"llm items: {fmt_items(own)} ; paragraph llm items: {fmt_items(para_items)}")
        rendered = [f"{i}: {reasons[(i, sid)]}" for i in sorted(own, key=_item_sort_key) if (i, sid) in reasons]
        rationale_text.append(" ; ".join(rendered) if rendered else "rationale: none")
    return {g: np.stack(v) for g, v in blocks.items()}, pseudo_text, graph_text, rationale_text


def build_features(ws: Workspace):
    """Adds ws.blocks and the text columns pseudo_text / graph_text / rationale_text to ws.frame."""
    records = {a: rec for recs in ws.llm_records.values() for a, rec in recs.items()}
    blocks = {g: [] for g in ["self", "context", "graph", "extras", "rationale"]}
    texts = {"pseudo_text": [], "graph_text": [], "rationale_text": []}
    for article, sub in tqdm(ws.frame.groupby("article", sort=False), desc="features"):
        idx = sub.index.to_numpy()
        b, pt, gt, rt = article_features(sub, ws.emb[idx], records[article])
        for g in blocks:
            blocks[g].append(b[g])
        texts["pseudo_text"] += pt
        texts["graph_text"] += gt
        texts["rationale_text"] += rt
    ws.blocks = {g: np.concatenate(v) for g, v in blocks.items()}
    for c, v in texts.items():
        ws.frame[c] = v
    return ws


def feature_matrix(ws, groups, mask):
    if not groups:
        return np.zeros((int(mask.sum()), 0), dtype=np.float32)
    return np.concatenate([ws.blocks[g][mask] for g in groups], axis=1).astype(np.float32)

In [15]:
build_features(ws)
print({g: b.shape[1] for g, b in ws.blocks.items()}, "feature dims per group")
ex = ws.frame[(ws.frame.split == "train") & (ws.frame.rationale_text != "rationale: none")
              & (ws.frame.graph_text != "graph evidence: none")].iloc[3]
print(f"\nExample ({ex.article} {ex.sid}, {ex.section}; gold {ex.gold_labels}):\n  {ex.text}")
print("  pseudo   :", ex.pseudo_text)
print("  rationale:", ex.rationale_text[:300])
print("  graph    :", ex.graph_text[:400])

features:   0%|          | 0/50 [00:00<?, ?it/s]

{'self': 70, 'context': 105, 'graph': 105, 'extras': 53, 'rationale': 140} feature dims per group

Example (PMC3002766 S45, Methods; gold ['3a']):
  This double-blind, randomised, parallel-group, placebo-controlled study consisted of a ≤2-week screening period, a 24-week treatment period and an 8-week post-treatment follow-up period.
  pseudo   : llm items: 3a ; paragraph llm items: none
  rationale: 3a: The sentence identifies a double-blind, randomised, parallel-group, placebo-controlled study and its study periods.
  graph    : [3a EXPECTED_ALLOCATION_IN 13a | sim 0.44] All 188 randomised patients received study treatment, with mean exposures of 22.7±0.9 and 24.5±0.9 weeks in the bosentan and placebo groups, respectively. ; [3a EXPECTED_ALLOCATION_IN 13a | sim 0.41] The treatment period was completed by 75/98 (76.5%) and 73/90 (81.1%) patients, respectively (figure 1), with 9.2% and 7.8%, respectively, discontinued due t


# Part C — Step 3: the residual classifier

PubMedBERT encodes

`[CLS] prev [SEP] section + target [SEP] next [SEP] pseudo-labels [SEP] rationale [SEP] graph [SEP]`

and an MLP encodes the structured groups. The output for item *i* is

`logit_i = prior_i(LLM label) + correction_i(x)`

- **`prior_i`** is the log-odds of gold *i* given whether the LLM said *i*. It is estimated on the training articles only, smoothed toward the pooled LLM rates, and floored at P = 0.55 when the LLM says *i*. So the untrained model reproduces the LLM's decisions.
- **`correction_i`** starts at zero and is L2-penalised. Frequent items learn corrections; rare items stay near the LLM. A version without this head lost to the LLM on rare items.

**Training:** baseline hyper-parameters, 5 epochs fixed, 3 seeds averaged, trained on the 30 train articles. It predicts validation and test. **Train** articles are scored by 5-fold **cross-fitting**. Each run is saved per seed and fold in `outputs/runs/` with its `config.json`. Existing runs are re-used only if their config matches; missing runs are trained, which needs a GPU.


In [16]:
# ---- classifier ----
@dataclass
class ClassifierConfig(Config):
    """Baseline `Config` (PubMedBERT, lr 1e-5 / 1e-3, batch 4, dropout 0.1, threshold 0.5) + enrichment."""
    model_name: str = PUBMEDBERT
    feature_groups: tuple = ("self", "context", "graph", "rationale")
    text_segments: tuple = ("pseudo", "rationale", "graph")
    feature_hidden: int = 128
    extra_token_budget: int = 288
    residual_prior: bool = True
    residual_l2: float = 0.01
    prior_smoothing: float = 10.0
    prior_floor_llm1: float = 0.55
    epochs: int = 5


def encode_enriched(row, tokenizer, cfg):
    """[CLS] prev [SEP] section+target [SEP] next [SEP] pseudo [SEP] rationale [SEP] graph [SEP].
    The baseline part keeps the baseline truncation rule; appended segments share `extra_token_budget`."""
    if not cfg.text_segments:
        return encode_row(row, tokenizer, cfg)[0]
    extras = [str(row[f"{s}_text"]) for s in ("pseudo", "rationale", "graph") if s in cfg.text_segments]
    budget, kept = cfg.extra_token_budget, []
    for t in extras:
        ids = tokenizer.encode(t, add_special_tokens=False)[:max(0, budget)]
        budget -= len(ids)
        kept.append(ids)
    tokens, _ = encode_row(row, tokenizer, replace(cfg, max_length=cfg.max_length - sum(len(x) + 1 for x in kept)))
    ids = list(tokens["input_ids"])
    for x in kept:
        ids += x + [tokenizer.sep_token_id]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "token_type_ids": [0] * len(ids)}


def item_prior_logits(ws, train_mask, smoothing, floor_llm1):
    """Per-item log-odds of gold given the LLM label, from TRAINING rows only, smoothed toward pooled rates.
    P(gold | LLM says item) is floored at `floor_llm1`, so the model starts exactly at the LLM decisions."""
    y, llm = ws.Y[train_mask], ws.llm_label[train_mask]
    n1, tp, n0, fn = llm.sum(0), (llm & y).sum(0), (~llm).sum(0), (~llm & y).sum(0)
    g1, g0 = tp.sum() / max(1, n1.sum()), fn.sum() / max(1, n0.sum())
    p1 = np.clip(np.maximum((tp + smoothing * g1) / (n1 + smoothing), floor_llm1), 1e-4, 1 - 1e-4)
    p0 = np.clip((fn + smoothing * g0) / (n0 + smoothing), 1e-4, 1 - 1e-4)
    logit = lambda p: np.log(p / (1 - p))
    return logit(p1).astype(np.float32), logit(p0).astype(np.float32)


def build_classifier(cfg, n_features):
    """PubMedBERT [CLS] (+ MLP over structured features) -> 34 logits; in residual mode the head is
    zero-initialised and its output is added to the LLM prior (passed as the last 34 feature columns)."""
    import torch
    from transformers import AutoConfig, AutoModel

    class Classifier(torch.nn.Module):
        def __init__(self):
            super().__init__()
            config = AutoConfig.from_pretrained(cfg.model_name, revision=cfg.revision)
            config.hidden_dropout_prob = config.attention_probs_dropout_prob = cfg.dropout
            self.encoder = AutoModel.from_pretrained(cfg.model_name, revision=cfg.revision, config=config)
            self.dropout = torch.nn.Dropout(cfg.dropout)
            self.feature_net = torch.nn.Sequential(torch.nn.Linear(n_features, cfg.feature_hidden), torch.nn.ReLU(),
                                                   torch.nn.Dropout(cfg.dropout)) if n_features else None
            self.classifier = torch.nn.Linear(self.encoder.config.hidden_size + (cfg.feature_hidden if n_features else 0),
                                              len(LABELS))
            if cfg.residual_prior:
                torch.nn.init.zeros_(self.classifier.weight)
                torch.nn.init.zeros_(self.classifier.bias)

        def forward(self, features=None, **batch):
            prior = None
            if cfg.residual_prior:
                features, prior = features[:, :-len(LABELS)], features[:, -len(LABELS):]
            h = self.dropout(self.encoder(**batch).last_hidden_state[:, 0, :])
            if self.feature_net is not None:
                h = torch.cat([h, self.feature_net(features)], dim=-1)
            correction = self.classifier(h)
            return (prior + correction, correction) if prior is not None else (correction, None)

    return Classifier()


def train_and_predict(ws, cfg, seed, train_articles, out_dir):
    """Train on `train_articles`, predict all other rows -> (n, 34) with NaN on training rows. Cached."""
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer
    path, cfg_path = Path(out_dir) / "probs.npy", Path(out_dir) / "config.json"
    cfg = replace(cfg, seed=seed)
    run_cfg = {**asdict(cfg), "train_articles": sorted(set(train_articles))}
    if path.exists():
        saved = json.loads(cfg_path.read_text()) if cfg_path.exists() else None
        assert saved == json.loads(json.dumps(run_cfg)), (
            f"{out_dir} was trained with a different config; delete that folder to retrain")
        return np.load(path)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(cfg.model_name, revision=cfg.revision)
    tr = ws.frame["article"].isin(set(train_articles)).to_numpy()
    te = ~tr
    X_tr, X_te = feature_matrix(ws, cfg.feature_groups, tr), feature_matrix(ws, cfg.feature_groups, te)
    n_feat = X_tr.shape[1]
    if cfg.residual_prior:
        l1, l0 = item_prior_logits(ws, tr, cfg.prior_smoothing, cfg.prior_floor_llm1)
        prior = lambda m: np.where(ws.llm_label[m], l1[None, :], l0[None, :]).astype(np.float32)
        X_tr, X_te = np.concatenate([X_tr, prior(tr)], 1), np.concatenate([X_te, prior(te)], 1)

    def loader(mask, X, shuffle):
        fr, y = ws.frame[mask].reset_index(drop=True), ws.Y[mask].astype(np.float32)
        rows = [{**encode_enriched(r, tok, cfg), "features": X[i], "labels": y[i]} for i, (_, r) in enumerate(fr.iterrows())]

        def collate(b):
            inputs = tok.pad([{k: r[k] for k in ("input_ids", "attention_mask", "token_type_ids")} for r in b],
                             return_tensors="pt")
            return (inputs, torch.tensor(np.stack([r["features"] for r in b])),
                    torch.tensor(np.stack([r["labels"] for r in b])))
        return DataLoader(rows, batch_size=cfg.batch_size, shuffle=shuffle, collate_fn=collate,
                          generator=torch.Generator().manual_seed(cfg.seed))

    model = build_classifier(cfg, n_feat).to(device)
    head = [p for n_, p in model.named_parameters() if not n_.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": model.encoder.parameters(), "lr": cfg.encoder_lr},
                             {"params": head, "lr": cfg.head_lr}], weight_decay=cfg.weight_decay)
    bce = torch.nn.BCEWithLogitsLoss()
    train_loader = loader(tr, X_tr, True)
    for _ in tqdm(range(cfg.epochs), desc=f"train seed {seed}"):
        model.train()
        for inputs, feats, y in train_loader:
            opt.zero_grad(set_to_none=True)
            logits, corr = model(features=feats.to(device), **{k: v.to(device) for k, v in inputs.items()})
            loss = bce(logits, y.to(device)) + (cfg.residual_l2 * corr.pow(2).mean() if corr is not None else 0.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    out = []
    with torch.no_grad():
        for inputs, feats, _ in loader(te, X_te, False):
            out.append(torch.sigmoid(model(features=feats.to(device), **{k: v.to(device) for k, v in inputs.items()})[0]).cpu().numpy())
    probs = np.full((len(ws.frame), len(LABELS)), np.nan, dtype=np.float32)
    probs[te] = np.concatenate(out)
    np.save(path, probs)
    save_json(run_cfg, cfg_path)
    return probs


CLF_SEEDS, CROSSFIT_FOLDS, CROSSFIT_SEEDS, CROSSFIT_FOLD_SEED = [42, 43, 44], 5, [42], 42


def crossfit_folds(ws):
    train = sorted(ws.frame.loc[ws.mask("train"), "article"].unique())
    shuffled = list(train)
    random.Random(CROSSFIT_FOLD_SEED).shuffle(shuffled)
    return train, [shuffled[i::CROSSFIT_FOLDS] for i in range(CROSSFIT_FOLDS)]


def classifier_probs(ws, cfg=None, run_dir=None):
    """Validation/test: mean of CLF_SEEDS models trained on all 30 train articles.
    Train: cross-fitted (5 folds inside train), so every row is predicted by a model that never saw its article.
    Cached runs are loaded; missing runs are trained (GPU recommended)."""
    cfg = cfg or ClassifierConfig()
    run_dir = Path(run_dir or ws.paths.classifier_cache / "runs" / "enriched_full_residual")
    train, folds = crossfit_folds(ws)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        probs = np.nanmean([train_and_predict(ws, cfg, s, train, run_dir / "train_only" / f"seed{s}") for s in CLF_SEEDS], 0)
        for k, held in enumerate(folds):
            m = ws.frame["article"].isin(held).to_numpy()
            cf = np.nanmean([train_and_predict(ws, cfg, s, sorted(set(train) - set(held)), run_dir / f"crossfit{k}" / f"seed{s}")
                             for s in CROSSFIT_SEEDS], 0)
            probs[m] = cf[m]
    ws.probs = probs
    return probs

In [18]:
classifier_probs(ws)
m = ws.mask("validation")
display(pd.DataFrame([{"validation": n, **{k: pair_metrics(p, ws.Y[m], ws.llm_label[m], ws.article_arr[m])[k] for k in ["P", "R", "F1", "macro F1"]}}
                      for n, p in [("LLM alone", ws.llm_label[m]), ("classifier (p >= 0.5)", ws.probs[m] >= 0.5)]]).set_index("validation").round(3))
print(f"Gold pairs inside the classifier's top-3 items per sentence (validation): "
      f"{100 * (np.argsort(np.argsort(-ws.probs, 1), 1)[m] < 3)[ws.Y[m]].mean():.1f}%")

,P,R,F1,macro F1
validation,,,,
LLM alone,0.690,0.821,0.750,0.635
classifier (p >= 0.5),0.792,0.703,0.745,0.609


Gold pairs inside the classifier's top-3 items per sentence (validation): 93.2%


# Part D — Step 4: triage

The classifier decides **which pairs deserve a second look**. Pairs where the LLM and the classifier agree are accepted without an LLM call. See the glossary for `remove?`, `add?` L1/L2 and the **superset**. The verifier is run once on the superset, and every stricter setting in `GRID` is scored offline in Part G.


In [19]:
# ---- triage ----
TOP_K = 3
SUPERSET = {"keep_below": 0.5, "add_at_least": 0.5, "add_unlabelled_at_least": 0.5}
GRID = {
    "keep_below": [0.1, 0.2, 0.3, 0.4, 0.5],
    "add_at_least": [0.5, 0.6, 0.7, 0.8, 1.01],                    # 1.01 = level-1 additions off
    "add_unlabelled_at_least": [0.5, 0.6, 0.7, 0.8, 0.9, 1.01],    # 1.01 = level 2 off
    "confidence_gate": [("high",), ("high", "medium")],
}


def topk_mask(ws, k=TOP_K):
    rank = np.argsort(np.argsort(-np.nan_to_num(ws.probs, nan=-1.0), axis=1), axis=1)
    return rank < k


def triage_flags(ws, keep_below, add_at_least, add_unlabelled_at_least, k=TOP_K):
    """remove?  : LLM said the item, classifier does not (not in top-k or p < keep_below)
       add? L1  : sentence already has an LLM label; classifier says a new item (top-k, p >= add_at_least)
       add? L2  : sentence has no LLM label; classifier says an item (top-k, p >= add_unlabelled_at_least)"""
    p = np.nan_to_num(ws.probs, nan=0.0)
    L, topk = ws.llm_label, topk_mask(ws, k)
    has = ~np.isnan(ws.probs).any(axis=1)[:, None]
    labelled = L.any(axis=1)[:, None]
    remove = L & ~(topk & (p >= keep_below)) & has
    add1 = labelled & ~L & topk & (p >= add_at_least) & has
    add2 = ~labelled & topk & (p >= add_unlabelled_at_least) & has
    return remove, add1, add2

In [20]:
rem, a1, a2 = triage_flags(ws, **SUPERSET)
rows = []
for split in ["train", "validation"]:
    s = ws.mask(split)[:, None]
    L, Y = ws.llm_label & s, ws.Y & s
    n = ws.frame.loc[ws.mask(split), "article"].nunique()
    rows.append({"split": split, "remove?": int((rem & s).sum()), "add? L1": int((a1 & s).sum()), "add? L2": int((a2 & s).sum()),
                 "flagged / article": round(((rem | a1 | a2) & s).sum() / n, 1),
                 "LLM FPs flagged %": round(100 * (rem & s & ~Y).sum() / (L & ~Y).sum(), 1),
                 "remove? truly wrong %": round(100 * (rem & s & ~Y).sum() / (rem & s).sum(), 1),
                 "LLM misses flagged %": round(100 * ((a1 | a2) & s & Y).sum() / (~L & Y).sum(), 1),
                 "add? truly gold %": round(100 * ((a1 | a2) & s & Y).sum() / ((a1 | a2) & s).sum(), 1)})
display(pd.DataFrame(rows).set_index("split"))

,remove?,add? L1,add? L2,flagged / article,LLM FPs flagged %,remove? truly wrong %,LLM misses flagged %,add? truly gold %
split,,,,,,,,
train,767,24,59,28.3,40.1,60.0,4.9,34.9
validation,303,2,6,31.1,51.7,61.1,1.1,25.0


# Part E — Step 5: the paragraph-level verifier

There is one LLM call per paragraph with flagged pairs. The prompt has six parts:
1. **Task spec (frozen):** the review protocol, how to use the guidance, the confidence levels, and the JSON schema.
2. **Item guidance:** the **same material the Step 1 extractors received**, restricted to the items in play (flagged items, the LLM's items in the paragraph, the classifier's top-3). That means definitions, guideline rules, global rules, item prototypes, pairwise boundary rules and worked examples.
3. **Corpus evidence**, from train gold only, with leave-one-article-out for train articles:
   - item statistics;
   - 2 annotated examples and 2 **near-misses** per flagged item, retrieved by embedding similarity with a bonus for the same section.
4. **Article memory** from Step 1, as context.
5. **The paragraph:** sentences with their current labels, plus the neighbouring paragraphs.
6. **Flagged pairs**, each `remove?` or `add?`, with the classifier probability (`SHOW_CLASSIFIER_PROB = True`). The task spec tells the verifier to treat it as evidence about the corpus convention, not as the answer.

**Output per pair:** `keep`/`remove` or `add`/`reject`, a confidence, a rule basis and a reason. Malformed or unknown decisions leave the label unchanged.

**Why corpus evidence:** without it (v33), the verifier kept 85% of the true false positives it was shown. It re-derived the extractor's reading of the rules, while the annotators draw narrower boundaries for items like 22, 18, 19, 17b and 20.


In [21]:
# ---- verifier task spec (frozen; the prompt signature hashes it) ----
VERIFIER_TASK_SPEC_BASE = """You are a CONSORT annotation REVIEWER. An upstream extractor has already labelled the sentences of one paragraph
with CONSORT checklist items ("current_items"). A triage step flagged some (sentence, item) pairs for a second look:

- check = "remove?": the extractor assigned this item; decide whether it should be REMOVED.
    decision: "keep" (the sentence satisfies the item under the guidance) or "remove" (it does not).
- check = "add?": the extractor did NOT assign this item; decide whether it should be ADDED.
    decision: "add" (the sentence satisfies the item under the guidance) or "reject" (it does not).

Judge ONLY the flagged pairs. Unflagged labels are context and stay as they are.

How to judge:
1. Apply the supplied guidance as the corpus definition of each item. It is the same guidance the extractor used:
   official definitions, annotation-guideline rules, global rules, item prototypes, pairwise boundary rules,
   and worked examples. Do not substitute generic CONSORT intuition for these corpus rules.
2. A sentence supports an item only if it explicitly reports what the item requires. Neighbouring sentences
   and the article memory may clarify what a sentence refers to, but they do not supply the missing content.
3. When two items compete for a sentence, use the pairwise boundary rules. A sentence may carry several items
   only if it explicitly satisfies each one.
4. Bare section headers carry no item.

Confidence:
- "high": the guidance clearly decides the case.
- "medium": the guidance leans one way.
- "low": unclear.
Low and medium decisions may be ignored downstream, so do not inflate confidence.

Return valid JSON only, no markdown:
{"decisions": [{"review_id": "Q0001", "decision": "keep|remove|add|reject", "confidence": "high|medium|low",
  "rule_basis": "<the definition/rule/boundary rule you relied on, short>", "reason": "<one sentence>"}]}
Every review_id must appear exactly once. Never invent review ids."""

VERIFIER_SPEC_PROB = """

classifier_probability (on each flagged pair): the probability, from a supervised model trained on this corpus's
gold annotations, that annotators labelled this sentence with this item. It reflects how the annotators actually
drew item boundaries in similar sentences, which can be narrower or broader than the written guidance. Treat it as
evidence about the corpus convention, weigh it against the text and the guidance, and say in "reason" when it
changed your decision. It is an estimate and is sometimes wrong."""

VERIFIER_SPEC_EVIDENCE = """

CORPUS EVIDENCE (at the end of item_guidance) comes from the gold annotations of OTHER articles in this corpus:
item statistics, sentences annotators labelled with an item, and near-misses: sentences where an extractor
following the same guidance assigned the item but the annotators did NOT (with what they assigned instead).
The annotations define the target labels. Where the written guidance and the annotated examples suggest different
boundaries, follow the boundary the annotated examples show. Mention in "rule_basis" when an example or statistic
decided the case."""

In [22]:
# ---- verifier ----
MAX_WORKED_EXAMPLES, ADJ_PARAGRAPH_CHARS = 5, 900
N_POSITIVE, N_NEAR_MISS, SAME_SECTION_BONUS, EXAMPLE_CHARS = 2, 2, 0.10, 300
SHOW_CLASSIFIER_PROB, USE_CORPUS_EVIDENCE, USE_ITEM_STATS = True, True, True
VERIFIER_MODEL = None          # None = the globally configured DSPy LM (gpt-5.6-terra in the reported run)

VERIFIER_TASK_SPEC = VERIFIER_TASK_SPEC_BASE + (VERIFIER_SPEC_PROB if SHOW_CLASSIFIER_PROB else "") + (
    VERIFIER_SPEC_EVIDENCE if (USE_CORPUS_EVIDENCE or USE_ITEM_STATS) else "")
PROMPT_SIG = hashlib.sha1(json.dumps({
    "spec": VERIFIER_TASK_SPEC, "model": VERIFIER_MODEL, "evidence": USE_CORPUS_EVIDENCE, "stats": USE_ITEM_STATS,
    "prob": SHOW_CLASSIFIER_PROB, "n_pos": N_POSITIVE, "n_neg": N_NEAR_MISS, "bonus": SAME_SECTION_BONUS, "top_k": TOP_K,
}, sort_keys=True).encode()).hexdigest()[:12]


def scope_snapshot(proto, items):
    """Original prototype restricted to `items`: item prototypes, pairwise rules inside the scope, worked examples."""
    items = {str(x) for x in items}
    item_prototypes = {i: proto.get("item_prototypes", {}).get(i, {})
                       for i in sorted(items, key=_item_sort_key) if i in proto.get("item_prototypes", {})}
    pairwise = {rid: r for rid, r in proto.get("pairwise_boundary_rules", {}).items()
                if {str(x) for x in r.get("items", [])} and {str(x) for x in r.get("items", [])} <= items}
    examples = [ex for exs in proto.get("worked_examples", {}).values() for ex in exs
                if {str(x) for x in ex.get("correct_items", [])} & items]
    return {"item_prototypes": item_prototypes, "pairwise_boundary_rules": pairwise,
            "worked_examples": examples[:MAX_WORKED_EXAMPLES]}


def render_guidance(ws, items):
    """The same guidance the Step 1 extractors used, restricted to the items in scope."""
    items = sorted({str(x) for x in items if str(x) in ACTIVE_EXTRACTION_ITEMS}, key=_item_sort_key)
    return "\n\n".join([
        "ITEMS IN SCOPE: " + ", ".join(items),
        "OFFICIAL DEFINITIONS:\n" + render_item_definitions(items, ws.prototype),
        "ANNOTATION GUIDELINE:\n" + render_section_guideline_text(ws.guideline, items),
        "GLOBAL RULES:\n" + render_global_rules_text(ws.prototype),
        "PROTOTYPE MEMORY (original, as given to the extractors):\n"
        + json.dumps(scope_snapshot(ws.prototype, items), ensure_ascii=False, indent=1),
    ])


def article_reviews(ws, article, flags):
    """Flagged pairs of one article (row order, item order, then remove/add1/add2)."""
    rem, a1, a2 = flags
    out = []
    for r in np.where(ws.article_arr == article)[0]:
        sid, pnum = str(ws.frame.at[r, "sid"]), int(ws.frame.at[r, "pnum"])
        for j, item in enumerate(LABELS):
            for flag, check, level in [(rem, "remove?", 1), (a1, "add?", 1), (a2, "add?", 2)]:
                if flag[r, j]:
                    out.append({"row": int(r), "sid": sid, "pnum": pnum, "item": item, "check": check,
                                "level": level, "p": float(ws.probs[r, j])})
    return out


def review_signature(reviews):
    """Cache key: the flagged pairs + the prompt/model settings."""
    key = json.dumps(sorted((x["sid"], x["item"], x["check"]) for x in reviews)) + PROMPT_SIG
    return hashlib.sha1(key.encode()).hexdigest()


def paragraph_packet(ws, article, pnum, reviews, topk):
    """Verifier inputs for one paragraph (adds `review_id` to each review dict)."""
    rec = ws.llm_records[split_of_article(ws, article)][article]
    paragraphs = parse_article_paragraphs_json(ws.article_json[article])
    sub = ws.frame[(ws.frame["article"] == article) & (ws.frame["pnum"] == pnum)]
    labels_by_sid = {str(s): [l for l, on in zip(LABELS, ws.llm_label[r]) if on] for s, r in zip(sub["sid"], sub.index)}
    topk_by_sid = {str(s): [LABELS[j] for j in np.where(topk[r])[0]] for s, r in zip(sub["sid"], sub.index)}
    ordered = sorted(paragraphs)
    pos = ordered.index(pnum)
    neighbour = lambda q: compact_text(paragraphs[q]["paragraph_text"], ADJ_PARAGRAPH_CHARS) if q is not None else ""
    paragraph = {
        "pnum": pnum, "router_sections": router_map(rec).get(pnum, []),
        "previous_paragraph": neighbour(ordered[pos - 1] if pos > 0 else None),
        "sentences": [{"sid": s["sid"], "text": s["text"], "current_items": labels_by_sid.get(s["sid"], [])}
                      for s in paragraphs[pnum]["sentences"]],
        "next_paragraph": neighbour(ordered[pos + 1] if pos + 1 < len(ordered) else None),
    }
    review_items = []
    for k, x in enumerate(reviews, 1):
        item = {"review_id": f"Q{k:04d}", "sid": x["sid"], "item": x["item"], "check": x["check"]}
        if SHOW_CLASSIFIER_PROB:
            item["classifier_probability"] = round(x["p"], 3)
        review_items.append(item)
        x["review_id"] = item["review_id"]
    scope = {x["item"] for x in reviews}
    for s in paragraph["sentences"]:
        scope |= set(s["current_items"]) | set(topk_by_sid.get(s["sid"], []))
    return {"item_guidance": render_guidance(ws, scope) + corpus_evidence(ws, article, reviews),
            "paragraph_json": json.dumps(paragraph, ensure_ascii=False),
            "review_items_json": json.dumps(review_items, ensure_ascii=False),
            "article_context_json": rec.get("article_context_json") or "{}"}


# ---- corpus evidence: TRAIN gold only; leave-one-article-out when the verified article is itself in train ----
_STATS_CACHE = {}


def _evidence_pool(ws, article):
    return ws.mask("train") & (ws.article_arr != article)


def _section_norm(ws):
    if "_section_norm" not in ws.frame:
        ws.frame["_section_norm"] = ws.frame["section"].map(normalize_section_name)
    return ws.frame["_section_norm"].to_numpy()


def item_stats_text(ws, item, article):
    excluded = article if split_of_article(ws, article) == "train" else None
    key = (item, excluded)
    if key in _STATS_CACHE:
        return _STATS_CACHE[key]
    j, pool, sec_norm = LABELS.index(item), _evidence_pool(ws, excluded), _section_norm(ws)
    Y, L = ws.Y[pool, j], ws.llm_label[pool, j]
    n_gold, n_llm, n_both = int(Y.sum()), int(L.sum()), int((Y & L).sum())
    sec = pd.Series(sec_norm[pool][Y]).value_counts(normalize=True)
    sec_txt = ", ".join(f"{s} {100 * v:.0f}%" for s, v in sec.head(3).items()) or "n/a"
    instead = [", ".join(l for l, on in zip(LABELS, ws.Y[r]) if on) or "no item" for r in np.where(pool)[0][L & ~Y]]
    inst = pd.Series(instead).value_counts(normalize=True).head(3) if instead else pd.Series(dtype=float)
    inst_txt = "; ".join(f"{k} {100 * v:.0f}%" for k, v in inst.items()) or "n/a"
    text = (f"- {item}: annotators labelled {n_gold} training sentences {item} (sections: {sec_txt}). "
            f"When the extractor assigned {item}, annotators agreed {100 * safe_div(n_both, n_llm):.0f}% of the time "
            f"(n={n_llm}); otherwise they labelled the sentence: {inst_txt}. "
            f"The extractor found {100 * safe_div(n_both, n_gold):.0f}% of the annotated {item} sentences.")
    _STATS_CACHE[key] = text
    return text


def corpus_evidence(ws, article, reviews):
    """Item statistics + annotated examples + near-misses for the flagged items (appended to item_guidance)."""
    if not (USE_CORPUS_EVIDENCE or USE_ITEM_STATS):
        return ""
    pool, sec_norm = _evidence_pool(ws, article), _section_norm(ws)
    items = sorted({x["item"] for x in reviews}, key=_item_sort_key)
    lines = ["", "CORPUS EVIDENCE (gold annotations of other training articles; never this article)"]
    if USE_ITEM_STATS:
        lines += ["Item statistics:"] + [item_stats_text(ws, i, article) for i in items]
    if USE_CORPUS_EVIDENCE:
        render = lambda r: (f"    - [{sec_norm[r]}; annotated: {', '.join(l for l, on in zip(LABELS, ws.Y[r]) if on) or 'no item'}] "
                            f"{compact_text(ws.frame.at[r, 'text'], EXAMPLE_CHARS)}")
        rows = np.where(pool)[0]
        for item in items:
            j = LABELS.index(item)
            target_rows = [x["row"] for x in reviews if x["item"] == item]
            sims = (ws.emb[pool] @ ws.emb[target_rows].T).max(axis=1)
            sims = sims + SAME_SECTION_BONUS * np.isin(sec_norm[pool], list({sec_norm[r] for r in target_rows}))
            pos, neg = rows[ws.Y[pool, j]], rows[ws.llm_label[pool, j] & ~ws.Y[pool, j]]
            rank = lambda cand: cand[np.argsort(-sims[np.searchsorted(rows, cand)])] if len(cand) else cand
            lines += [f"Item {item}:", f"  annotated as {item}:"]
            lines += [render(r) for r in rank(pos)[:N_POSITIVE]] or ["    - (none in training data)"]
            lines.append(f"  near-misses: extractor assigned {item}, annotators did NOT:")
            lines += [render(r) for r in rank(neg)[:N_NEAR_MISS]] or ["    - (none in training data)"]
    return "\n".join(lines)


def sanitize_decisions(raw, reviews):
    """Keep only valid decisions for known review ids; anything else leaves the label unchanged."""
    by_id = {x["review_id"]: x for x in reviews}
    allowed = {"remove?": {"keep", "remove"}, "add?": {"add", "reject"}}
    out, events = {}, []
    try:
        obj = extract_json_object(raw)
    except Exception as e:
        obj = {}
        events.append({"kind": "parse_error", "detail": repr(e)})
    for d in obj.get("decisions", []) or []:
        rid = str(d.get("review_id", "")).strip()
        if rid not in by_id:
            events.append({"kind": "unknown_review_id", "review_id": rid})
            continue
        x, decision = by_id[rid], str(d.get("decision", "")).strip().lower()
        if decision not in allowed[x["check"]]:
            events.append({"kind": "invalid_decision", "review_id": rid, "decision": decision})
            continue
        conf = str(d.get("confidence", "low")).strip().lower()
        out[f"{x['item']}|{x['sid']}"] = {
            "check": x["check"], "decision": decision, "confidence": conf if conf in {"high", "medium", "low"} else "low",
            "rule_basis": str(d.get("rule_basis", ""))[:300], "reason": str(d.get("reason", ""))[:400]}
    return out, events


def make_reviewer(lm=None):
    """DSPy module; the signature fields are the paragraph packet + the frozen task spec."""
    import dspy

    class ParagraphReview(dspy.Signature):
        """Re-judge flagged CONSORT (sentence, item) pairs in one paragraph, following fixed_task_spec and item_guidance."""
        article_id = dspy.InputField(desc="Article identifier")
        fixed_task_spec = dspy.InputField(desc="Frozen review protocol and output schema")
        item_guidance = dspy.InputField(desc="Definitions, guideline rules, global rules, prototype memory for items in scope")
        article_context_json = dspy.InputField(desc="Grounded article memory (context only, not evidence)")
        paragraph_json = dspy.InputField(desc="Paragraph with sentences, their current item labels, neighbouring paragraphs")
        review_items_json = dspy.InputField(desc="Flagged (sid, item, check) pairs to decide")
        output_json = dspy.OutputField(desc="JSON object with a decisions list")

    module = dspy.Predict(ParagraphReview)
    if lm is not None:
        module.set_lm(lm)
    return module


def verify_article(ws, article, flags, topk, lm=None):
    """One LLM call per paragraph with flagged pairs -> cache record for the article."""
    reviews = article_reviews(ws, article, flags)
    module = make_reviewer(lm)
    by_pnum = defaultdict(list)
    for x in reviews:
        by_pnum[x["pnum"]].append(x)
    decisions, events, calls = {}, [], 0
    for pnum, rv in sorted(by_pnum.items()):
        packet = paragraph_packet(ws, article, pnum, rv, topk)
        pred = module(article_id=str(article), fixed_task_spec=VERIFIER_TASK_SPEC, **packet)
        calls += 1
        got, ev = sanitize_decisions(getattr(pred, "output_json", pred), rv)
        decisions.update(got)
        events += [{**e, "pnum": pnum} for e in ev]
    return {"article": article, "signature": review_signature(reviews), "n_reviews": len(reviews),
            "calls": calls, "decisions": decisions, "events": events}


def load_verifier_cache(path):
    out = {}
    path = Path(path)
    if path.exists():
        for line in open(path, encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                out[r["article"]] = r
    return out


def run_verifier(ws, split, cache_dir, lm=None, n_workers=4):
    """Verify every article of `split` whose cached record is missing or stale; appends to `<cache_dir>/<split>.jsonl`."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    flags, topk = triage_flags(ws, **SUPERSET), topk_mask(ws)
    path = Path(cache_dir) / f"{split}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    done = load_verifier_cache(path)
    arts = sorted(ws.frame.loc[ws.mask(split), "article"].unique())
    todo = [a for a in arts if a not in done or done[a]["signature"] != review_signature(article_reviews(ws, a, flags))]
    if todo:
        with ThreadPoolExecutor(max_workers=n_workers) as pool, open(path, "a", encoding="utf-8") as f:
            futures = {pool.submit(verify_article, ws, a, flags, topk, lm): a for a in todo}
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"verifier [{split}]"):
                r = fut.result()
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                done[r["article"]] = r
    return done

In [23]:
flags, topk = triage_flags(ws, **SUPERSET), topk_mask(ws)
VERIFIER_DIR = OUTPUT_DIR / "verifier"
for split in RUN_VERIFIER_SPLITS:
    assert split != "test", "the test split is verified in Part H (RUN_TEST), after the validation selection"
    run_verifier(ws, split, VERIFIER_DIR, n_workers=N_WORKERS)
decisions = {s: load_verifier_cache(VERIFIER_DIR / f"{s}.jsonl") for s in ["train", "validation", "test"]}
for s, d in decisions.items():
    if d:
        ok = sum(r["signature"] == review_signature(article_reviews(ws, a, flags)) for a, r in d.items())
        print(f"{s:10s}: {len(d)} articles, {sum(r['calls'] for r in d.values())} calls, "
              f"{sum(r['n_reviews'] for r in d.values())} flagged pairs; cache matches current prompt: {ok}/{len(d)}")
print("prompt signature:", PROMPT_SIG)

validation: 10 articles, 151 calls, 311 flagged pairs; cache matches current prompt: 10/10
prompt signature: 1ec0afe8d377


### A real prompt and its decisions

Here is one validation paragraph with both `remove?` and `add?` checks: the parts of the prompt, its size, and the cached decisions next to gold.

In [24]:
for a in sorted(decisions["validation"]):
    by_pnum = defaultdict(list)
    for x in article_reviews(ws, a, flags):
        by_pnum[x["pnum"]].append(x)
    cand = [p for p, rv in by_pnum.items() if {x["check"] for x in rv} == {"remove?", "add?"} and len(rv) <= 4]
    if cand:
        article, pnum = a, cand[0]
        break
rv = by_pnum[pnum]
packet = paragraph_packet(ws, article, pnum, rv, topk)
guidance, evidence = packet["item_guidance"].split("\nCORPUS EVIDENCE", 1)
print(f"=== {article}, paragraph {pnum}: {len(rv)} flagged pairs; prompt ≈ "
      f"{(len(VERIFIER_TASK_SPEC) + sum(len(v) for v in packet.values())) // 4} tokens ===")
print("\n--- item guidance (first 600 of", len(guidance), "chars) ---\n" + guidance[:600] + "\n  ...")
print("\n--- CORPUS EVIDENCE" + evidence)
print("\n--- paragraph (truncated) ---\n" + json.dumps(json.loads(packet["paragraph_json"]), indent=1, ensure_ascii=False)[:1200])
print("\n--- flagged pairs -> decisions ---")
for x in json.loads(packet["review_items_json"]):
    d = decisions["validation"][article]["decisions"].get(f"{x['item']}|{x['sid']}", {})
    gold = x["item"] in ws.frame.loc[(ws.frame.article == article) & (ws.frame.sid == x["sid"]), "gold_labels"].iloc[0]
    print(f"{x['review_id']} {x['sid']} {x['item']:>4} {x['check']:8s} -> {d.get('decision')} ({d.get('confidence')}) | gold: "
          f"{'yes' if gold else 'no'}\n      basis: {d.get('rule_basis')}\n      reason: {d.get('reason')}")

=== PMC4797126, paragraph 57: 4 flagged pairs; prompt ≈ 10264 tokens ===

--- item guidance (first 600 of 28403 chars) ---
ITEMS IN SCOPE: 6a, 17a, 18, 22

OFFICIAL DEFINITIONS:
- 6a: Completely defined pre-specified primary and secondary outcome measures, including how and when they were assessed
- 17a: For each primary and secondary outcome, results for each group, and the estimated effect size and its precision
- 18: Results of any other analyses performed, including subgroup analyses and adjusted analyses, distinguishing pre-specified from exploratory
- 22: Interpretation consistent with results, balancing benefits and harms, and considering other relevant evidence

ANNOTATION GUIDELINE:
General annotation rul
  ...

--- CORPUS EVIDENCE (gold annotations of other training articles; never this article)
Item statistics:
- 17a: annotators labelled 450 training sentences 17a (sections: results 100%). When the extractor assigned 17a, annotators agreed 63% of the time (n=560); otherwise 

# Part F — Prompt development on train (optional)

Use this loop to improve the verifier prompt **without touching validation**:
1. Edit the task spec or the evidence settings, and give the version a new `DEV_LABEL`.
2. Run the verifier on a fixed subset of train articles. These use cross-fitted probabilities, and their evidence excludes the article itself.
3. Score it at a fixed setting and log it.
4. Compare with the previous version: how many decisions flipped, how many flips fixed vs broke a pair, per item, with examples.

At temperature 1, some flips are just sampling noise. Trust a clearly lopsided fixed-vs-broke split more than a small F1 difference.


In [25]:
DEV_LABEL = "v34-rebuild-default"                 # a NEW label for every prompt version
DEV_N_ARTICLES, DEV_SEED = 10, 0           # fixed, seeded subset of the 30 train articles
DEV_SETTING = {**SUPERSET, "confidence_gate": ("high", "medium")}   # held fixed while developing
DEV_DIR = OUTPUT_DIR / "prompt_dev"

if RUN_PROMPT_DEV:
    train_articles = sorted(ws.frame.loc[ws.mask("train"), "article"].unique())
    dev_articles = sorted(random.Random(DEV_SEED).sample(train_articles, min(DEV_N_ARTICLES, len(train_articles))))
    run_dir = DEV_DIR / f"{re.sub(r'[^A-Za-z0-9_.-]+', '-', DEV_LABEL)}__{PROMPT_SIG}"
    run_dir.mkdir(parents=True, exist_ok=True)
    cache_path = run_dir / "train.jsonl"
    done = load_verifier_cache(cache_path)
    todo = [a for a in dev_articles if a not in done or done[a]["signature"] != review_signature(article_reviews(ws, a, flags))]
    with ThreadPoolExecutor(N_WORKERS) as pool, open(cache_path, "a", encoding="utf-8") as f:
        for fut in tqdm(as_completed([pool.submit(verify_article, ws, a, flags, topk) for a in todo]), total=len(todo), desc="dev"):
            r = fut.result()
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            done[r["article"]] = r

    in_dev = ws.frame.loc[ws.mask("train"), "article"].isin(dev_articles).to_numpy()   # dev rows within train
    dev_pred = lambda dec: apply_policy(ws, "train", DEV_SETTING, "verifier", dec)[in_dev]
    tr = ws.mask("train")
    Yd, Ld, Ad = ws.Y[tr][in_dev], ws.llm_label[tr][in_dev], ws.article_arr[tr][in_dev]
    pred = dev_pred(done)
    m_ = pair_metrics(pred, Yd, Ld, Ad)
    log_path = DEV_DIR / "prompt_dev_log.csv"
    log = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame(columns=["label"])
    log = pd.concat([log[log["label"] != DEV_LABEL], pd.DataFrame([{
        "label": DEV_LABEL, "prompt_sig": PROMPT_SIG, "dir": str(run_dir), "articles": len(dev_articles),
        **{k: round(m_[k], 4) for k in ["P", "R", "F1", "FP corrected %", "TP lost %", "misses recovered %"]},
        "LLM F1": round(pair_metrics(Ld, Yd, Ld, Ad)["F1"], 4)}])], ignore_index=True)
    log.to_csv(log_path, index=False)
    display(log.drop(columns=["dir"]).set_index("label"))

    prev = log[log["label"] != DEV_LABEL].tail(1)          # diff against the previous version
    if len(prev):
        prev_pred = dev_pred(load_verifier_cache(Path(prev["dir"].iloc[0]) / "train.jsonl"))
        r_, a1_, a2_ = triage_flags(ws, DEV_SETTING["keep_below"], DEV_SETTING["add_at_least"], DEV_SETTING["add_unlabelled_at_least"])
        flagged = (r_ | a1_ | a2_)[tr][in_dev]
        flips = flagged & (prev_pred != pred)
        fixed, broke = flips & (pred == Yd), flips & (pred != Yd)
        print(f"vs '{prev['label'].iloc[0]}': {int(flips.sum())} decisions flipped -> {int(fixed.sum())} fixed, "
              f"{int(broke.sum())} broke (net {int(fixed.sum()) - int(broke.sum()):+d})")
        by_item = pd.DataFrame({"fixed": fixed.sum(0), "broke": broke.sum(0)}, index=LABELS)
        display(by_item[(by_item.fixed + by_item.broke) > 0].T)
else:
    print("Prompt development is off (RUN_PROMPT_DEV = False).")

Prompt development is off (RUN_PROMPT_DEV = False).


# Part G — Validation: selection, comparison, worked examples

**Selection.** Grid search over `GRID` (three thresholds × gate) for the best validation micro-F1, separately for the verifier and for "classifier decides", which gets **its own** selected setting. The result is saved to `selection.json`. Because the selection was made on these same numbers, they are slightly optimistic; the test run in Part H is the estimate.

**Rows of the policy table:**
- **LLM alone:** Step 1 output, unchanged.
- **classifier alone (p ≥ 0.5):** the classifier's own labels for every pair, with no triage.
- **classifier decides (selected):** LLM labels, with the flagged pairs decided by the classifier, at its selected setting.
- **verifier @ superset:** LLM labels, with every superset flag decided by the verifier (two gates).
- **verifier v34 (selected):** LLM labels, with the flagged pairs decided by the verifier, at its selected setting and gate.


In [26]:
dec_val = decisions["validation"]
assert dec_val, "no validation decisions: run the verifier (RUN_VERIFIER_SPLITS = ['validation'])"
sel_verifier = select_setting(ws, dec_val, "verifier")
sel_classifier = select_setting(ws, dec_val, "classifier")
save_json({"verifier": sel_verifier, "classifier_decides": sel_classifier, "prompt_sig": PROMPT_SIG}, OUTPUT_DIR / "selection.json")
print("selected (verifier):  ", sel_verifier)
print("selected (classifier):", sel_classifier)

table = policy_table(ws, "validation", [
    ("verifier @ superset, gate=high", {**SUPERSET, "confidence_gate": ("high",)}, "verifier"),
    ("verifier @ superset, gate=high+medium", {**SUPERSET, "confidence_gate": ("high", "medium")}, "verifier"),
    ("classifier decides (selected)", sel_classifier, "classifier"),
    ("verifier v34 (selected)", sel_verifier, "verifier")], decisions=dec_val)
table.to_csv(OUTPUT_DIR / "validation_policies.csv")
COLS = ["P", "R", "F1", "macro F1", "FP corrected %", "TP lost %", "misses recovered %", "removal precision %",
        "addition precision %", "removed", "added", "presence fixed vs LLM", "presence broken vs LLM", "reviews"]
display(table[COLS])

selected (verifier):   {'keep_below': 0.5, 'add_at_least': 0.6, 'add_unlabelled_at_least': 0.5, 'confidence_gate': ('high', 'medium')}
selected (classifier): {'keep_below': 0.2, 'add_at_least': 0.6, 'add_unlabelled_at_least': 0.7, 'confidence_gate': ('high',)}


,P,R,F1,macro F1,FP corrected %,TP lost %,misses recovered %,removal precision %,addition precision %,removed,added,presence fixed vs LLM,presence broken vs LLM,reviews
policy,,,,,,,,,,,,,,
LLM alone,0.690,0.821,0.750,0.635,0.000,0.000,0.000,0.000,0.000,0,0,0,0,NaN
classifier alone (p>=0.5),0.792,0.703,0.745,0.609,51.676,14.662,1.149,61.258,25.000,302,8,10,13,0.0
"verifier @ superset, gate=high",0.719,0.799,0.757,0.632,15.642,2.882,1.149,70.886,66.667,79,3,3,1,311.0
"verifier @ superset, gate=high+medium",0.735,0.795,0.764,0.635,22.905,3.383,1.149,75.229,50.000,109,4,4,1,311.0
classifier decides (selected),0.722,0.805,0.761,0.632,15.922,2.130,0.575,77.027,100.000,74,1,4,1,75.0
verifier v34 (selected),0.736,0.795,0.765,0.635,22.905,3.383,1.149,75.229,66.667,109,3,4,1,309.0


**Per check type.** The fraction of flagged pairs where each side's verdict was right, at the superset with the high+medium gate:

In [27]:
def check_accuracy(dec, label):
    o = outcome_table(ws, "validation", {**SUPERSET, "confidence_gate": ("high", "medium")}, dec)
    o["check type"] = o["check"].str[:7].str.strip()
    return o.groupby("check type").agg(pairs=("gold", "size"), **{label: ("verifier_right", "mean")}), o

acc, outcomes = check_accuracy(dec_val, "verifier v34")
acc["classifier"] = outcomes.groupby("check type")["classifier_right"].mean()
display(acc.round(3))
print("Items where v34 kept a false positive (top):", outcomes[outcomes.outcome == "FP kept"].item.value_counts().head(6).to_dict())
print("\nPer item (flagged pairs, superset, high+medium gate): share of pairs each side got right")
by_item = outcomes.groupby("item").agg(n=("gold", "size"), verifier_acc=("verifier_right", "mean"),
                                       classifier_acc=("classifier_right", "mean"))
display(by_item[by_item.n >= 5].sort_values("n", ascending=False).round(3))

,pairs,verifier v34,classifier
check type,,,
add? L1,2,0.500,0.000
add? L2,6,0.833,0.333
remove?,303,0.571,0.611


Items where v34 kept a false positive (top): {'17a': 26, '17b': 24, '12b': 12, '18': 9, '16': 8, '22': 7}

Per item (flagged pairs, superset, high+medium gate): share of pairs each side got right


,n,verifier_acc,classifier_acc
item,,,
17a,89,0.618,0.629
17b,41,0.341,0.732
22,35,0.686,0.457
18,30,0.700,0.567
12b,28,0.393,0.714
16,19,0.579,0.579
6a,14,0.857,0.143
20,10,0.400,0.800
13b,9,0.667,0.444


### Worked examples

Every flagged pair at the selected setting gets one of eight outcomes:
- `remove?` pairs: **FP corrected**, **FP kept**, **TP wrongly removed**, **TP correctly kept**;
- `add?` pairs: **miss recovered**, **miss not recovered**, **wrong addition**, **correctly rejected**.

Each example shows the sentence and its neighbours, the LLM / gold / classifier top-3 labels, and the verifier's decision with the rule it cited. Gold appears only to show whether a change was right.

In [28]:
o = outcome_table(ws, "validation", sel_verifier, dec_val)
display(pd.crosstab(o["check"], o["outcome"], margins=True))
for group in ["FP corrected", "miss recovered", "TP wrongly removed", "FP kept"]:
    g = o[o.outcome == group]
    print("=" * 25, f"{group}: {len(g)} pairs (2 examples)", "=" * 25)
    for _, x in g.sample(min(2, len(g)), random_state=0).iterrows():
        show_example(ws, x)

outcome,FP corrected,FP kept,TP correctly kept,TP wrongly removed,correctly rejected,miss recovered,wrong addition,All
check,,,,,,,,
add? L2,0,0,0,0,3,2,1,6
remove?,82,103,91,27,0,0,0,303
All,82,103,91,27,3,2,1,309


========================= FP corrected: 82 pairs (2 examples) =========================
[FP corrected] PMC5009486 S117 | item 17b | remove? | verifier: remove (high) | classifier p=0.10
  SENTENCE: Fig. 4 Time to onset of action (≥10 % improvement from baseline FEV1) on Day 1 (mITT population).
prev: The proportion of patients who had achieved onset of action by 15 min post-dose ranged from 7 % [...]
next: Cumulative data are shown.
  section: Results | LLM: 17a, 17b | gold: 17a | classifier top-3: 17a 0.88, 17b 0.10, 18 0.02
  rule basis: 17b not_if: do not assign to a caption or display label lacking a binary-result effect; 17a__17b
    boundary requires explicit absolute and relative effects.
  reason: This figure caption defines and names the onset outcome but reports no binary-result estimate or
    paired absolute and relative effect.

[FP corrected] PMC4770816 S117 | item 17a | remove? | verifier: remove (medium) | classifier p=0.21
  SENTENCE: One major cardiovascular event occ

In [29]:
# Change log for one article: every applied change and every correction the verifier declined
ARTICLE = o["article"].value_counts().index[0]
mark = {"FP corrected": "✓ removed", "TP wrongly removed": "✗ removed", "miss recovered": "✓ added", "wrong addition": "✗ added",
        "FP kept": "· kept (was FP)", "miss not recovered": "· rejected (was gold)", "TP correctly kept": "  kept",
        "correctly rejected": "  rejected"}
lg = o[o.article == ARTICLE].assign(n=lambda d: d.sid.str[1:].astype(int)).sort_values(["n", "item"])
print(f"{ARTICLE}: {len(lg)} flagged pairs, {int(lg.applied.sum())} applied changes\n")
for sid, g in lg.groupby("sid", sort=False):
    print(textwrap.shorten(f"{sid}: {ws.frame.at[int(g.row.iloc[0]), 'text']}", 130))
    for _, x in g.iterrows():
        print(f"    {x['item']:>4} {x['check']:<8} {mark[x['outcome']]:<22} {x['decision']}/{x['confidence']:<6} p={x['p']:.2f}  "
              f"{textwrap.shorten(str(x['reason']), 70)}")

PMC5264229: 65 flagged pairs, 19 applied changes

S63: A post hoc analysis of simplified disease activity index (SDAI) and clinical disease activity index (CDAI) was done to [...]
      6a add? L2  ✓ added                add/high   p=0.50  The sentence defines the post hoc efficacy measure as the [...]
S78: To determine equivalence between SB2 and INF the 95% CI of the ACR20 rate difference had to be within the prespecified [...]
     12a remove?  · kept (was FP)        keep/high   p=0.32  The sentence specifies the prespecified confidence-interval [...]
S82: The primary efficacy outcome was analysed using the PPS and the full analysis set (FAS).22
     12b remove?  ✓ removed              remove/medium p=0.10  The sentence only states that the primary efficacy outcome used [...]
S84: In addition, if missing data occurred, such patients were assumed to be ACR20 non-responders in FAS.
     12b remove?  · kept (was FP)        keep/high   p=0.19  Assuming missing ACR20 data to be non-respo

# Part BASE — baselines, and what each step does (with counts)

**Baselines** (same split, same 34 items, same sentence × item metric as everything above):
1. **LLM only:** the same LLM reads the whole article and labels sentences using **only the official CONSORT 2010 checklist definitions** (`inputs/CONSORT_2010_Checklist.csv`). There is no router, prototype, annotation guideline, corpus evidence, classifier or verifier. One call per article.
2. **PubMedBERT text-only:** the early supervised baseline: previous / target / next sentence with section names, and **no LLM input at all**. Same 30 train articles, seeds and epochs as the residual classifier.

**Accounting:** at the **selected** verifier setting, every LLM false positive, correct pair and miss is followed through the system:
- did the classifier flag it;
- what would "classifier decides" do with it;
- what did the verifier do with it.

The same accounting is then broken down **per item**. The functions take a split, so Part H repeats all of this on test.

In [30]:
# ---- baseline 1: LLM only (whole article + checklist definitions) ----
BASELINE_DIR = OUTPUT_DIR / "baselines"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

_checklist = pd.read_csv(INPUT_DIR / "CONSORT_2010_Checklist.csv")
LLM_ONLY_SPEC = """You annotate a randomised controlled trial report with CONSORT 2010 checklist items.
Read the whole article (paragraphs with numbered sentences). For every sentence that reports a checklist item, output
the item id and the sentence id. A sentence may report several items; most sentences report none.

CONSORT 2010 checklist items:
""" + "\n".join(f"- {str(r['Item No']).strip()} ({r['Item Name']}): {r['Description']}"
                for _, r in _checklist.iterrows() if str(r["Item No"]).strip() in LABELS) + """

Return valid JSON only, no markdown: {"extractions": [{"item": "17a", "sid": "S12"}]}
Use only the item ids listed above and copy sentence ids exactly."""


class LLMOnlyExtraction(dspy.Signature):
    """Label the sentences of a trial report with CONSORT 2010 checklist items, following task_spec."""
    article_id = dspy.InputField(desc="Article identifier")
    task_spec = dspy.InputField(desc="Task, item definitions and output format")
    article_json = dspy.InputField(desc="Paragraphs with numbered sentences (sid, text)")
    output_json = dspy.OutputField(desc="JSON object with an extractions list")


def compact_article(article_json):
    """Paragraphs with their sentences only (no paragraph text, router output or memory)."""
    paras = parse_article_paragraphs_json(article_json)
    return json.dumps([{"pnum": p, "sentences": paras[p]["sentences"]} for p in sorted(paras)], ensure_ascii=False)


def run_llm_only(ws, split):
    """One call per article; results appended to outputs/baselines/llm_only_<split>.jsonl (finished articles skipped)."""
    path = BASELINE_DIR / f"llm_only_{split}.jsonl"
    done = load_verifier_cache(path)
    todo = [a for a in sorted(ws.frame.loc[ws.mask(split), "article"].unique()) if a not in done]
    module = dspy.Predict(LLMOnlyExtraction)

    def one(a):
        pred = module(article_id=str(a), task_spec=LLM_ONLY_SPEC, article_json=compact_article(ws.article_json[a]))
        return {"article": str(a), "output_json": getattr(pred, "output_json", "")}

    if todo:
        with ThreadPoolExecutor(N_WORKERS) as pool, open(path, "a", encoding="utf-8") as f:
            for fut in tqdm(as_completed([pool.submit(one, a) for a in todo]), total=len(todo), desc=f"LLM only [{split}]"):
                r = fut.result()
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                done[r["article"]] = r
    return done


def llm_only_matrix(ws, split, records):
    """(rows of `split`) x 34 labels from the LLM-only outputs; unknown items / sids are ignored."""
    frame = ws.frame[ws.mask(split)]
    row_of = {(a, str(x)): k for k, (a, x) in enumerate(zip(frame["article"], frame["sid"]))}
    out, bad = np.zeros((len(frame), len(LABELS)), dtype=bool), 0
    for a, r in records.items():
        try:
            exts = extract_json_object(r["output_json"]).get("extractions", []) or []
        except Exception:
            exts, bad = [], bad + 1
        for e in exts:
            item, sid = str(e.get("item", "")).strip(), str(e.get("sid", "")).strip()
            if item in LABELS and (a, sid) in row_of:
                out[row_of[(a, sid)], LABELS.index(item)] = True
    if bad:
        print(f"[{split}] {bad} article output(s) were not valid JSON (counted as no extractions)")
    return out


llm_only_records = {}
if RUN_BASELINES:
    llm_only_records["validation"] = run_llm_only(ws, "validation")
    print(f"[validation] LLM-only outputs: {len(llm_only_records['validation'])} articles")

[validation] LLM-only outputs: 10 articles


In [31]:
# ---- baseline 2: PubMedBERT text-only (prev / target / next sentence + section names; no LLM input) ----
SUP_CONFIG = ClassifierConfig(feature_groups=(), text_segments=(), residual_prior=False)   # same epochs/seeds as Part C


def text_only_probs(ws):
    train = sorted(ws.frame.loc[ws.mask("train"), "article"].unique())
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean([train_and_predict(ws, SUP_CONFIG, s, train, OUTPUT_DIR / "runs" / "baseline_text_only" / "train_only" / f"seed{s}")
                           for s in CLF_SEEDS], 0)


sup_probs = text_only_probs(ws) if RUN_BASELINES else None
print("text-only baseline:", "trained / loaded" if sup_probs is not None else "off",
      f"| epochs {SUP_CONFIG.epochs}, seeds {CLF_SEEDS}")

text-only baseline: trained / loaded | epochs 5, seeds [42, 43, 44]


In [32]:
# ---- system comparison: baselines -> Step 1 -> classifier -> verifier ----
def system_predictions(ws, split, decisions_split):
    s = ws.mask(split)
    systems = {}
    if split in llm_only_records:
        systems["LLM only (checklist definitions)"] = llm_only_matrix(ws, split, llm_only_records[split])
    if sup_probs is not None:
        systems["PubMedBERT text-only (p>=0.5)"] = np.nan_to_num(sup_probs[s]) >= 0.5
    systems["Step 1 extractor"] = ws.llm_label[s]
    systems["classifier alone (p>=0.5)"] = np.nan_to_num(ws.probs[s]) >= 0.5
    systems["Step 1 + classifier decides (selected)"] = apply_policy(ws, split, sel_classifier, "classifier")
    systems["Step 1 + verifier (selected)"] = apply_policy(ws, split, sel_verifier, "verifier", decisions_split)
    return systems


def compare_systems(ws, split, systems):
    s = ws.mask(split)
    Y, L, arts = ws.Y[s], ws.llm_label[s], ws.article_arr[s]
    table = pd.DataFrame([{"system": name, "pairs predicted": int(p.sum()),
                           **{k: v for k, v in pair_metrics(p, Y, L, arts).items() if k in ["P", "R", "F1", "macro F1", "item-presence acc %"]}}
                          for name, p in systems.items()]).set_index("system").round(3)
    print(f"{split}: {len(np.unique(arts))} articles, {int(Y.sum())} gold pairs")
    display(table)
    table.to_csv(BASELINE_DIR / f"system_comparison_{split}.csv")
    others = [n for n in systems if n != "Step 1 extractor"]
    boot = article_bootstrap(ws, split, systems, [(n, "Step 1 extractor") for n in others])
    display(boot.set_index("comparison"))
    return table


systems_val = system_predictions(ws, "validation", dec_val)
compare_val = compare_systems(ws, "validation", systems_val)

validation: 10 articles, 972 gold pairs


,pairs predicted,P,R,F1,macro F1,item-presence acc %
system,,,,,,
LLM only (checklist definitions),943,0.596,0.578,0.587,0.538,86.471
PubMedBERT text-only (p>=0.5),919,0.737,0.697,0.716,0.601,89.118
Step 1 extractor,1156,0.690,0.821,0.750,0.635,89.412
classifier alone (p>=0.5),862,0.792,0.703,0.745,0.609,88.529
Step 1 + classifier decides (selected),1083,0.722,0.805,0.761,0.632,90.294
Step 1 + verifier (selected),1050,0.736,0.795,0.765,0.635,90.294


,Δ F1,95% CI low,95% CI high,P(Δ ≤ 0),articles improved
comparison,,,,,
LLM only (checklist definitions) − Step 1 extractor,-0.1631,-0.1990,-0.1341,1.0000,0/10
PubMedBERT text-only (p>=0.5) − Step 1 extractor,-0.0340,-0.0752,0.0032,0.9582,3/10
classifier alone (p>=0.5) − Step 1 extractor,-0.0052,-0.0233,0.0126,0.7098,5/10
Step 1 + classifier decides (selected) − Step 1 extractor,0.0111,0.0031,0.0201,0.0012,6/10
Step 1 + verifier (selected) − Step 1 extractor,0.0146,0.0075,0.0205,0.0000,9/10


In [33]:
# ---- what each step does: follow every LLM error / correct pair through the classifier and the verifier ----
def error_flow(ws, split, decisions_split):
    """At the selected verifier setting. 'classifier decides' is applied to the SAME flagged pairs, for contrast."""
    s = ws.mask(split)
    Y, L = ws.Y[s], ws.llm_label[s]
    rem, a1, a2 = (m[s] for m in triage_flags(ws, sel_verifier["keep_below"], sel_verifier["add_at_least"],
                                              sel_verifier["add_unlabelled_at_least"]))
    flag_remove, flag_add = rem, a1 | a2
    clf = apply_policy(ws, split, {**sel_verifier, "confidence_gate": ("high",)}, "classifier")
    ver = apply_policy(ws, split, sel_verifier, "verifier", decisions_split)
    FP, TP, FN, NEG = L & ~Y, L & Y, ~L & Y, ~L & ~Y
    n = lambda x: int(x.sum())
    flow = pd.DataFrame([
        {"LLM outcome": "false positive (LLM said item, gold no)", "flag": "remove?", "LLM count": n(FP),
         "flagged by classifier": n(FP & flag_remove), "classifier decides: fixed": n(FP & ~clf),
         "verifier: fixed": n(FP & ~ver), "still wrong after verifier": n(FP & ver)},
        {"LLM outcome": "correct (LLM said item, gold yes)", "flag": "remove?", "LLM count": n(TP),
         "flagged by classifier": n(TP & flag_remove), "classifier decides: fixed": 0,
         "verifier: fixed": 0, "still wrong after verifier": n(TP & ~ver)},
        {"LLM outcome": "miss (gold item, LLM did not say it)", "flag": "add?", "LLM count": n(FN),
         "flagged by classifier": n(FN & flag_add), "classifier decides: fixed": n(FN & clf),
         "verifier: fixed": n(FN & ver), "still wrong after verifier": n(FN & ~ver)},
        {"LLM outcome": "not gold, not said (correctly empty)", "flag": "add?", "LLM count": n(NEG),
         "flagged by classifier": n(NEG & flag_add), "classifier decides: fixed": 0,
         "verifier: fixed": 0, "still wrong after verifier": n(NEG & ver)},
    ]).set_index("LLM outcome")
    print(f"{split} | selected verifier setting: {sel_verifier}")
    print("Rows 2 and 4 are damage: 'still wrong' there counts correct pairs the verifier removed and empty pairs it added.")
    display(flow)
    print(f"\nOn the same {n(flag_remove)} remove? and {n(flag_add)} add? flags:")
    for name, final in [("classifier decides", clf), ("verifier", ver)]:
        fixed, broke = n(FP & ~final) + n(FN & final), n(TP & ~final) + n(NEG & final)
        print(f"  {name:19s}: fixes {n(FP & ~final)} FPs + {n(FN & final)} misses, breaks {n(TP & ~final)} correct pairs "
              f"+ adds {n(NEG & final)} wrong pairs -> net {fixed - broke:+d}")
    print(f"  coverage: the classifier flags {100 * n(FP & flag_remove) / max(1, n(FP)):.0f}% of LLM FPs and "
          f"{100 * n(FN & flag_add) / max(1, n(FN)):.0f}% of LLM misses; unflagged errors can only be fixed by Step 1 itself")
    flow.to_csv(BASELINE_DIR / f"error_flow_{split}.csv")
    return {"FP": FP, "TP": TP, "FN": FN, "NEG": NEG, "flag_remove": flag_remove, "flag_add": flag_add, "ver": ver}


flow_val = error_flow(ws, "validation", dec_val)

validation | selected verifier setting: {'keep_below': 0.5, 'add_at_least': 0.6, 'add_unlabelled_at_least': 0.5, 'confidence_gate': ('high', 'medium')}
Rows 2 and 4 are damage: 'still wrong' there counts correct pairs the verifier removed and empty pairs it added.


,flag,LLM count,flagged by classifier,classifier decides: fixed,verifier: fixed,still wrong after verifier
LLM outcome,,,,,,
"false positive (LLM said item, gold no)",remove?,358,185,185,82,276
"correct (LLM said item, gold yes)",remove?,798,118,0,0,27
"miss (gold item, LLM did not say it)",add?,174,2,2,2,172
"not gold, not said (correctly empty)",add?,68948,4,0,0,1



On the same 303 remove? and 6 add? flags:
  classifier decides : fixes 185 FPs + 2 misses, breaks 118 correct pairs + adds 4 wrong pairs -> net +65
  verifier           : fixes 82 FPs + 2 misses, breaks 27 correct pairs + adds 1 wrong pairs -> net +56
  coverage: the classifier flags 52% of LLM FPs and 1% of LLM misses; unflagged errors can only be fixed by Step 1 itself


In [34]:
# ---- per item: F1 of every system, and what each step did ----
SHORT = {"LLM only (checklist definitions)": "F1 LLM only", "PubMedBERT text-only (p>=0.5)": "F1 PubMedBERT",
         "Step 1 extractor": "F1 Step 1", "classifier alone (p>=0.5)": "F1 classifier",
         "Step 1 + classifier decides (selected)": "F1 +clf decides", "Step 1 + verifier (selected)": "F1 +verifier"}


def per_item_breakdown(ws, split, systems, f):
    def item_f1(p, y):
        tp, fp, fn = (p & y).sum(0), (p & ~y).sum(0), (~p & y).sum(0)
        return np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(1, 2 * tp + fp + fn), np.nan)

    Y = ws.Y[ws.mask(split)]
    t = pd.DataFrame(index=LABELS)
    t["gold"] = Y.sum(0)
    for name, p in systems.items():
        t[SHORT[name]] = item_f1(p, Y)
    t["LLM FP"] = f["FP"].sum(0)
    t["FP flagged"] = (f["FP"] & f["flag_remove"]).sum(0)
    t["FP fixed (verifier)"] = (f["FP"] & ~f["ver"]).sum(0)
    t["TP wrongly removed"] = (f["TP"] & ~f["ver"]).sum(0)
    t["LLM miss"] = f["FN"].sum(0)
    t["miss flagged"] = (f["FN"] & f["flag_add"]).sum(0)
    t["miss fixed (verifier)"] = (f["FN"] & f["ver"]).sum(0)
    t["wrong adds"] = (f["NEG"] & f["ver"]).sum(0)
    t = t[t["gold"] + t["LLM FP"] > 0].sort_values("gold", ascending=False)
    display(t.round(3))
    t.to_csv(BASELINE_DIR / f"per_item_breakdown_{split}.csv")
    f1_cols = [c for c in t.columns if c.startswith("F1")]
    best = t[f1_cols].fillna(-1)
    print("Items where each system is best (ties counted for all):",
          {c: int((best[c] >= best.max(axis=1) - 1e-9).sum()) for c in f1_cols})
    return t


per_item_val = per_item_breakdown(ws, "validation", systems_val, flow_val)

,gold,F1 LLM only,F1 PubMedBERT,F1 Step 1,F1 classifier,F1 +clf decides,F1 +verifier,LLM FP,FP flagged,FP fixed (verifier),TP wrongly removed,LLM miss,miss flagged,miss fixed (verifier),wrong adds
6a,155,0.548,0.784,0.864,0.824,0.864,0.864,16,1,0,1,25,1,1,0
17a,131,0.634,0.732,0.757,0.766,0.801,0.805,76,56,30,6,5,0,0,1
22,130,0.520,0.703,0.777,0.742,0.779,0.787,43,15,8,4,20,1,1,0
5,57,0.750,0.775,0.883,0.860,0.883,0.862,5,1,0,2,8,0,0,0
12a,55,0.614,0.750,0.720,0.739,0.720,0.721,25,5,2,1,10,0,0,0
17b,41,0.192,0.548,0.472,0.431,0.469,0.454,40,30,6,3,16,0,0,0
15,35,0.615,0.827,0.899,0.899,0.899,0.899,3,0,0,0,4,0,0,0
18,33,0.400,0.259,0.522,0.355,0.533,0.571,35,17,8,0,9,0,0,0
7a,30,0.983,0.846,0.846,0.846,0.846,0.846,0,0,0,0,8,0,0,0
19,30,0.527,0.676,0.706,0.706,0.706,0.706,14,1,0,0,6,0,0,0


Items where each system is best (ties counted for all): {'F1 LLM only': 5, 'F1 PubMedBERT': 5, 'F1 Step 1': 13, 'F1 classifier': 13, 'F1 +clf decides': 13, 'F1 +verifier': 17}


# Part H — Test (run once)

Set `RUN_TEST = True` after the selection in Part G is final, and run this part. It:
1. runs the verifier on the test articles, at the superset, with the frozen prompt;
2. runs the LLM-only baseline on test (the text-only baseline already predicts test);
3. writes `test_lock.json` the first time. Re-running after the selection changes stops with an error;
4. reports the policy table, the article-level paired bootstrap (5,000 resamples), the system comparison, the error accounting and the per-item breakdown.

In [35]:
if not RUN_TEST:
    print("Test is off (RUN_TEST = False).")
else:
    frozen = json.loads((OUTPUT_DIR / "selection.json").read_text())
    assert frozen["prompt_sig"] == PROMPT_SIG, "the verifier prompt changed after selection"
    lock = OUTPUT_DIR / "test_lock.json"
    if lock.exists():
        assert json.loads(lock.read_text())["selection"] == frozen, "selection changed after the test run"
    else:
        save_json({"selection": frozen, "time": str(pd.Timestamp.now())}, lock)
    dec_test = decisions["test"] = run_verifier(ws, "test", VERIFIER_DIR, n_workers=N_WORKERS)
    if RUN_BASELINES:
        llm_only_records["test"] = run_llm_only(ws, "test")
    display(policy_table(ws, "test", [("classifier decides (selected)", sel_classifier, "classifier"),
                                      ("verifier v34 (selected)", sel_verifier, "verifier")], decisions=dec_test)[COLS])
    systems_test = system_predictions(ws, "test", dec_test)
    compare_test = compare_systems(ws, "test", systems_test)
    flow_test = error_flow(ws, "test", dec_test)
    per_item_test = per_item_breakdown(ws, "test", systems_test, flow_test)

Test is off (RUN_TEST = False).


# Part I — Prototype leakage check, history, next steps

This check looks for validation or test sentences among the prototype's boundary-rule examples, which carry gold items. With the train-only prototype the count should be zero.

In [36]:
def prototype_leakage(ws, min_chars=60, prefix_chars=120):
    """Verbatim matches between non-train sentences and the prototype's pairwise boundary-rule examples.
    Each example carries its gold item, so a match exposes that sentence's label to every prompt that includes the rule."""
    norm = lambda t: re.sub(r"\s+", " ", str(t).lower()).strip()
    examples = []
    for rid, rule in ws.prototype.get("pairwise_boundary_rules", {}).items():
        for e in rule.get("examples", []):
            examples.append((rid, e.get("item"), norm(e.get("sentence", ""))))
    rows = []
    for r, (a, sid, text, split) in enumerate(zip(ws.frame["article"], ws.frame["sid"], ws.frame["text"], ws.frame["split"])):
        key = norm(text)[:prefix_chars]
        if len(str(text)) <= min_chars:
            continue
        for rid, item, ex in examples:
            if key in ex:
                rows.append({"row": r, "split": split, "article": a, "sid": sid, "rule": rid, "example item": item,
                             "gold": ", ".join(l for l, on in zip(LABELS, ws.Y[r]) if on) or "none"})
                break
    return pd.DataFrame(rows)


leak = prototype_leakage(ws)
display(leak.groupby("split").agg(sentences=("sid", "size"), articles=("article", "nunique")))
leaked = set(leak["row"])
rows = []
for split in ["validation", "test"]:
    for flag in (True, False):
        sel = [r for r in np.where(ws.mask(split))[0] if (r in leaked) == flag and ws.Y[r].any()]
        tp, fn = (ws.llm_label[sel] & ws.Y[sel]).sum(), (~ws.llm_label[sel] & ws.Y[sel]).sum()
        rows.append({"split": split, "in prototype": flag, "gold-labelled sentences": len(sel), "LLM recall": round(tp / max(1, tp + fn), 3)})
display(pd.DataFrame(rows).set_index(["split", "in prototype"]))

,sentences,articles
split,,
train,233,30


gold-labelled sentences  LLM recall
split      in prototype                                     
validation True                                0       0.000
           False                             892       0.821
test       True                                0       0.000
           False                             947       0.761

### Development history

| version | change | result (validation unless noted) |
|---|---|---|
| v26 | LLM extractor, old prototype | F1 0.690; the old prototype held gold sentences from validation and test articles |
| baseline | PubMedBERT text-only, trained on the 30 train articles | 0.716 |
| v29–v31 | classifier input + LLM pseudo-labels, context, graph, rationale | small gains over the LLM |
| v32 | LLM-prior residual head | 5-fold CV 0.729 vs LLM 0.690 |
| v33 | triage + verifier with the extractor's rules | 0.703; the verifier mostly re-confirmed the extractor |
| v34 | + corpus evidence from train gold | 0.727, with far fewer changes than "classifier decides" |
| **v34, rebuilt prototype** | train-only prototype; Step 1 verifier removed; extractor rationales; classifier probability shown to the verifier | **Step 1 0.750 → verifier 0.765** (classifier decides 0.761; LLM only 0.587) |

Tried and dropped: strong + context graph edges in the classifier input, and graph-linked sentences in the verifier prompt. Neither improved validation F1.

### Next steps
- **Deciding is the bigger gap.** A perfect decider on the current flags would reach about 0.82 F1 (Part BASE). Combine the verifier and the classifier per item: the verifier is better on 6a, 22 and 18; the classifier on 17b, 12a, 12b and 20 (Part G per-item table). Fit the combination on cross-fitted **train** flags, not on validation.
- **Train the classifier as an error detector** for the LLM's labels: an LLM-positive pair is correct or not. Most false positives are co-assigned pairs (17a/17b, 17a/18, 12a/12b).
- **Recall must come from Step 1.** The classifier flags about 1% of LLM misses. Check which items carry the misses (per-item table), e.g. 14a, which the Methods extractor cannot output under the `official_only` item policy.